# Observation Resolution × Importance Sampling × Gradient Estimation
## Final research-grade Kaggle notebook — VAE / IWAE / DReG

This notebook implements the supplied frozen research plan after a second methodological audit.

### Scientific scope

- **RQ1:** effect of observation resolution `R ∈ {2,16,256}`.
- **RQ2:** effect and saturation of training importance samples `K ∈ {1,5,10,20,50}`.
- **RQ3:** standard IWAE vs. DReG-IWAE for `K>1`.
- **RQ4:** descriptive/paired-bootstrap `R × K × estimator` interaction patterns.
- **RQ5:** matched replication on **EMNIST-Balanced** after the complete MNIST main study is analyzed.
- **Latent dimension is hard-locked to `20` and is never swept.** The older optional latent-dimension sensitivity add-on is excluded because the latest study constraint supersedes it.

### Corrections incorporated from the audit

1. Pilot completion no longer implies pilot success: a **predeclared fail-closed pilot gate** checks late convergence, numerical stability, and gradient-spike diagnostics before the main grid is released.
2. MASTER CONFIG values are executable or explicitly validated; no decorative settings can silently disagree with the implementation.
3. The global best configuration is chosen by **validation bpd at common `K_eval=1000`**, never by test performance.
4. Final scientific analysis requires the **entire expected grid**, not merely whichever seed cells happened to finish.
5. Debug subsets cannot produce final findings.
6. Completed-run reuse is protected by source-spec, implementation, fixed-protocol, dataset, and software fingerprints.
7. Computational metrics are compared only across a uniform benchmark environment.
8. K=1 latency measures the complete density-computation path rather than only reconstruction.
9. Main-run timing removes per-batch gradient-norm synchronization; heavy gradient diagnostics are pilot-only.
10. Training, validation, and test memory peaks are measured separately.
11. Validation uses common random numbers across epochs to reduce Monte-Carlo checkpoint-selection noise.
12. Bootstrap CIs are produced for configuration means and paired contrasts, including explicit three-way interaction contrasts.
13. Convergence plots cover every configured K/estimator branch.
14. Convergence-speed summaries include epoch/time to the validation-selected checkpoint.
15. A predeclared descriptive saturation rule is implemented and clearly labeled as an operational reporting convention.
16. EMNIST analysis is matched directly to MNIST interaction patterns.
17. Adam backend options are pinned so timing/memory do not silently change with PyTorch defaults.
18. Multi-session Kaggle sharding/resume and epoch-level recovery are supported without overwriting completed experiments.

No result in this notebook is fabricated. Tables and findings are populated only when the corresponding experiments actually complete.

# 1. Experiment Overview

The authoritative design uses the same underlying byte image for all observation-resolution conditions. Each image is deterministically quantized to `R` levels, then modeled with the prescribed likelihood:

| R | Deterministic representation | Likelihood | Decoder output per pixel |
|---:|---|---|---:|
| 2 | 2 levels | Bernoulli | 1 logit |
| 16 | 16 levels | Categorical(16) | 16 logits |
| 256 | 256 levels | single discretized logistic | location + scale |

The common encoder/decoder trunk is `784 → 200 → 200 → z(20) → 200 → 200`; only the final likelihood head changes. Because the heads necessarily differ in parameter count, the notebook reports parameters, MACs, model bytes, and memory by `R` instead of assuming resolution has zero computational cost.

**Primary cross-condition metric:** bits per dimension (bpd). Final test results are evaluated at the same `K_eval=1000` for every trained configuration. Test data are never used for protocol freezing, checkpoint selection, or global configuration selection.

# 2. MASTER CONFIGURATION

Edit this cell to change experimental ranges, fixed protocol choices, phase toggles, benchmarking settings, sharding, and output paths. Experimental variables are the only quantities intentionally swept.

If the pilot rejects the current fixed protocol, change the relevant **fixed** value here and rerun; the notebook never silently auto-tunes from pilot or test outcomes.

In [ ]:
# =========================
# MASTER CONFIGURATION
# =========================
from pathlib import Path
import math

# ---------------------------------------------------------
# EXPERIMENTAL VARIABLES — intentionally varied
# ---------------------------------------------------------
EXPERIMENTAL = {
    "observation_resolutions": [2, 16, 256],
    "k_train_values": [1, 5, 10, 20, 50],
    "estimators_k_gt_1": ["iwae", "dreg"],
    "seeds": [1, 2, 3],
}

# ---------------------------------------------------------
# FIXED SCIENTIFIC / TRAINING PROTOCOL
# ---------------------------------------------------------
FIXED = {
    # Model
    "input_dim": 28 * 28,
    "hidden_dim": 200,
    "latent_dim": 20,                 # HARD LOCK; never experimental
    "activation": "tanh",
    "prior": "standard_normal",
    "posterior": "diagonal_gaussian",
    "weight_init": "xavier_uniform",
    "dtype": "float32",

    # Optimization
    "optimizer": "adam",
    "lr": 1e-3,
    "adam_betas": (0.9, 0.999),
    "adam_eps": 1e-4,
    "weight_decay": 0.0,
    "adam_amsgrad": False,
    "adam_foreach": False,            # pin backend; avoids version-dependent foreach auto-selection
    "adam_fused": False,
    "adam_maximize": False,
    "adam_capturable": False,
    "adam_differentiable": False,
    "adam_decoupled_weight_decay": False,
    "epochs": 100,
    "batch_size": 128,
    "lr_schedule": "constant",       # supported: constant | staged_50_75_x0.1
    "gradient_clip_norm": None,

    # Split discipline
    "validation_fraction": 1.0 / 6.0, # MNIST: 10k; EMNIST-Balanced: 18.8k
    "split_seed": 2026,

    # Evaluation / diagnostics
    "k_eval": 1000,
    "eval_k_chunk_size": 50,          # memory-only chunking; logsumexp remains mathematically exact
    "active_unit_threshold": 1e-2,
    "min_logistic_scale": math.exp(-7.0),

    # Statistics
    "bootstrap_repetitions": 10000,
    "bootstrap_seed": 424242,
    # Operational definition requested by RQ2: first K after which all subsequent mean adjacent gains
    # are <= 10% of the initial K=1 -> first-K>1 gain. Descriptive only, not a significance test.
    "saturation_relative_gain_fraction": 0.10,

    # DataLoader / numerical controls
    "num_workers": 2,
    "pin_memory": True,
    "persistent_workers": False,
    "deterministic_algorithms": True,
    "allow_tf32": False,

    # Benchmarking
    "benchmark_batch_size": 128,
    "benchmark_warmup": 5,
    "benchmark_repetitions": 20,
    "ram_poll_interval_sec": 0.20,
}

# ---------------------------------------------------------
# PREDECLARED PILOT ACCEPTANCE — pass/fail only, never auto-tunes
# ---------------------------------------------------------
PILOT = {
    "enabled": True,
    "resolutions": [2, 256],
    "k_values": [1, 10, 50],
    "estimators_k_gt_1": ["iwae", "dreg"],
    "seed": 1,
    "execute_on_shard_index": 0,

    # Operational gates, declared before outcomes are observed.
    "late_window_epochs": 10,
    "max_late_improvement_bpd": 0.005,  # larger continuing improvement => epoch budget not yet accepted
    "max_late_regression_bpd": 0.010,   # larger late worsening => LR schedule/stability not accepted
    "max_gradient_spike_ratio": 100.0,  # max batch grad norm / median batch grad norm
    "require_all_finite": True,
}

# ---------------------------------------------------------
# DATASET / PHASE TOGGLES
# ---------------------------------------------------------
PHASES = {
    "run_mnist_main_grid": True,
    "run_emnist_generalization_grid": True,
    "run_final_analysis": True,
}

# ---------------------------------------------------------
# KAGGLE EXECUTION / PERSISTENCE
# ---------------------------------------------------------
KAGGLE_WORKING = Path("/kaggle/working")
BASE_WORKING = KAGGLE_WORKING if KAGGLE_WORKING.exists() else Path.cwd()
RUN_CONTROL = {
    "data_root": str(BASE_WORKING / "data"),
    "output_root": str(BASE_WORKING / "latent_density_results"),
    "run_tag": "research_grade_z20_v2",
    "download_data": True,

    # Resume roots may point to attached read-only Kaggle datasets from previous sessions.
    # Example: ["/kaggle/input/my-prior-latent-density-results/research_grade_z20_v2"]
    "resume_roots": [],
    "resume_completed": True,
    "resume_incomplete": True,
    "continue_after_run_failure": True,

    # Sharding lets the 81/81-run grids be spread across Kaggle sessions without changing the design.
    "num_shards": 1,
    "shard_index": 0,

    # Debug limits apply only to the current execution shard. Any non-None value suppresses final findings.
    "debug_max_runs": None,

    # Scientific aggregation requires matching software fingerprints.
    "require_uniform_software_for_scientific_analysis": True,
    # Cost comparisons require one benchmark environment (GPU/software/backend) across compared runs.
    "require_uniform_benchmark_environment_for_efficiency": True,
}

# Immutable source-plan identity embedded at notebook-build time.
SOURCE_SPEC_SHA256 = "7e60cb2f5db157122552f477c9f06d082ba5d97b3801891254b2eec97ffd44f3"

print("MASTER CONFIG loaded")
print("Experimental variables:", EXPERIMENTAL)
print("Latent dimension (fixed):", FIXED["latent_dim"])

MASTER CONFIG loaded
Experimental variables: {'observation_resolutions': [2, 16, 256], 'k_train_values': [1, 5, 10, 20, 50], 'estimators_k_gt_1': ['iwae', 'dreg'], 'seeds': [1, 2, 3]}
Latent dimension (fixed): 20


# 3. EMNIST Dataset Selection

The frozen RQ5 protocol specifies **EMNIST-Balanced**. The split is exposed separately as requested so an EMNIST alphabet/task split can be selected without touching implementation code. A non-`balanced` choice is supported but is automatically recorded as a protocol deviation from the frozen RQ5 claim.

In [ ]:
# =========================
# EMNIST DATASET SELECTION
# =========================
EMNIST_CONFIG = {
    "split": "letters",  # byclass | bymerge | balanced | letters | digits | mnist
}
ALLOWED_EMNIST_SPLITS = {"byclass", "bymerge", "balanced", "letters", "digits", "mnist"}
assert EMNIST_CONFIG["split"] in ALLOWED_EMNIST_SPLITS
EMNIST_PROTOCOL_DEVIATION = EMNIST_CONFIG["split"] != "balanced"
if EMNIST_PROTOCOL_DEVIATION:
    print("WARNING: RQ5 in the frozen plan is specifically EMNIST-Balanced; this selection is a documented deviation.")
else:
    print("EMNIST selection matches frozen RQ5 protocol: balanced")

# 4. Environment & Reproducibility

The notebook records Python, package, CUDA, GPU, CPU/RAM, optimizer backend, and Kaggle-related environment information. PyTorch explicitly notes that exact bitwise reproducibility is not guaranteed across releases/platforms; this implementation therefore both seeds all relevant RNGs and fingerprints the software environment used for each run.

**Primary / authoritative references used by the fixed protocol**

- Kingma & Welling (2014), *Auto-Encoding Variational Bayes*, ICLR, arXiv:1312.6114.
- Burda, Grosse & Salakhutdinov (2016), *Importance Weighted Autoencoders*, ICLR, arXiv:1509.00519.
- Tucker, Lawson, Gu & Maddison (2019), *Doubly Reparameterized Gradient Estimators for Monte Carlo Objectives*, ICLR, arXiv:1810.04152.
- Glorot & Bengio (2010), *Understanding the difficulty of training deep feedforward neural networks*, AISTATS, PMLR 9:249–256.
- Kingma & Ba (2015), *Adam: A Method for Stochastic Optimization*, ICLR, arXiv:1412.6980.
- Salimans et al. (2017), *PixelCNN++*, ICLR, arXiv:1701.05517; official OpenAI PixelCNN++ implementation.
- Theis, van den Oord & Bethge (2016), *A note on the evaluation of generative models*, ICLR.
- Cohen et al. (2017), *EMNIST: Extending MNIST to handwritten letters*, IJCNN, DOI:10.1109/IJCNN.2017.7966217.
- PyTorch official reproducibility, DataLoader, Adam, and CUDA synchronization documentation.Primary-source links for auditability: [AEVB](https://arxiv.org/abs/1312.6114), [IWAE](https://arxiv.org/abs/1509.00519), [DReG](https://arxiv.org/abs/1810.04152), [Glorot & Bengio](https://proceedings.mlr.press/v9/glorot10a.html), [Adam](https://arxiv.org/abs/1412.6980), [PixelCNN++](https://arxiv.org/abs/1701.05517), [official OpenAI PixelCNN++ code](https://github.com/openai/pixel-cnn), [EMNIST/NIST](https://www.nist.gov/itl/products-and-services/emnist-dataset), and [PyTorch reproducibility](https://docs.pytorch.org/docs/stable/notes/randomness.html).

In [ ]:
import os
os.environ.setdefault("CUBLAS_WORKSPACE_CONFIG", ":4096:8")

import sys, json, time, random, platform, subprocess, hashlib, traceback, threading, inspect, types, shutil
from datetime import datetime, timezone
from dataclasses import dataclass, asdict
from collections import defaultdict
from pathlib import Path

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import psutil

import torch
import torch.nn as nn
import torch.nn.functional as F
from torch.utils.data import Dataset, DataLoader
import torchvision
from torchvision import datasets
from IPython.display import display
import importlib.metadata as importlib_metadata

DEVICE = torch.device("cuda" if torch.cuda.is_available() else "cpu")
DTYPE = torch.float32


def stable_seed(label: str, *parts) -> int:
    payload = "|".join([str(label), *[str(p) for p in parts]]).encode("utf-8")
    return int.from_bytes(hashlib.sha256(payload).digest()[:8], "little") % (2**63 - 1)


def set_global_seed(seed: int):
    seed = int(seed)
    random.seed(seed)
    np.random.seed(seed % (2**32))
    torch.manual_seed(seed)
    if torch.cuda.is_available():
        torch.cuda.manual_seed_all(seed)
    torch.backends.cudnn.benchmark = False
    torch.backends.cudnn.deterministic = True
    if hasattr(torch.backends, "cuda") and hasattr(torch.backends.cuda, "matmul"):
        torch.backends.cuda.matmul.allow_tf32 = bool(FIXED["allow_tf32"])
    if hasattr(torch.backends, "cudnn"):
        torch.backends.cudnn.allow_tf32 = bool(FIXED["allow_tf32"])
    torch.use_deterministic_algorithms(bool(FIXED["deterministic_algorithms"]), warn_only=False)


def safe_git_commit():
    try:
        return subprocess.check_output(["git", "rev-parse", "HEAD"], stderr=subprocess.DEVNULL, text=True).strip()
    except Exception:
        return None


def environment_snapshot():
    snap = {
        "timestamp_utc": datetime.now(timezone.utc).isoformat(),
        "python": sys.version,
        "python_version_short": platform.python_version(),
        "platform": platform.platform(),
        "processor": platform.processor(),
        "cpu_logical_count": psutil.cpu_count(logical=True),
        "cpu_physical_count": psutil.cpu_count(logical=False),
        "ram_total_gb": psutil.virtual_memory().total / (1024**3),
        "torch": torch.__version__,
        "torchvision": torchvision.__version__,
        "numpy": np.__version__,
        "pandas": pd.__version__,
        "matplotlib": importlib_metadata.version("matplotlib"),
        "psutil": psutil.__version__,
        "cuda_available": torch.cuda.is_available(),
        "cuda_runtime": torch.version.cuda,
        "cudnn_version": torch.backends.cudnn.version() if torch.cuda.is_available() else None,
        "device": str(DEVICE),
        "git_commit": safe_git_commit(),
        "kaggle_kernel_run_type": os.environ.get("KAGGLE_KERNEL_RUN_TYPE"),
        "kaggle_url_base": os.environ.get("KAGGLE_URL_BASE"),
    }
    if torch.cuda.is_available():
        try:
            snap["nvidia_driver"] = subprocess.check_output(["nvidia-smi","--query-gpu=driver_version","--format=csv,noheader"],text=True,stderr=subprocess.DEVNULL).splitlines()[0].strip()
        except Exception:
            snap["nvidia_driver"] = None
        props = torch.cuda.get_device_properties(0)
        snap.update({
            "gpu_name": props.name,
            "gpu_total_memory_gb": props.total_memory / (1024**3),
            "gpu_compute_capability": f"{props.major}.{props.minor}",
        })
    return snap


def sha256_json(obj):
    return hashlib.sha256(json.dumps(obj, sort_keys=True, default=str).encode("utf-8")).hexdigest()


def validate_configuration():
    errors=[]; warnings=[]
    if FIXED["latent_dim"] != 20: errors.append("latent_dim must remain exactly 20")
    if FIXED["input_dim"] != 784: errors.append("input_dim must remain 784 for 28x28 MNIST/EMNIST")
    if FIXED["activation"] != "tanh": errors.append("only the frozen tanh activation is implemented")
    if FIXED["prior"] != "standard_normal": errors.append("only standard_normal prior is implemented")
    if FIXED["posterior"] != "diagonal_gaussian": errors.append("only diagonal_gaussian posterior is implemented")
    if FIXED["weight_init"] != "xavier_uniform": errors.append("only xavier_uniform initialization is implemented")
    if FIXED["optimizer"] != "adam": errors.append("only frozen Adam optimizer is implemented")
    if FIXED["dtype"] != "float32": errors.append("only float32 protocol is implemented")
    if FIXED["lr_schedule"] not in {"constant", "staged_50_75_x0.1"}: errors.append("unsupported lr_schedule")
    for key in ["observation_resolutions","k_train_values","seeds"]:
        vals=EXPERIMENTAL[key]
        if not vals: errors.append(f"{key} cannot be empty")
        if len(vals) != len(set(vals)): errors.append(f"{key} contains duplicates")
    if not set(EXPERIMENTAL["observation_resolutions"]).issubset({2,16,256}): errors.append("R must be drawn from {2,16,256}")
    if any(int(k)<=0 for k in EXPERIMENTAL["k_train_values"]): errors.append("all K values must be positive")
    if 1 not in EXPERIMENTAL["k_train_values"]: warnings.append("K=1 VAE baseline is absent; this is not the frozen full design")
    if any(m not in {"iwae","dreg"} for m in EXPERIMENTAL["estimators_k_gt_1"]): errors.append("unsupported estimator")
    if not (0 < FIXED["validation_fraction"] < 0.5): errors.append("validation_fraction must be in (0,0.5)")
    if FIXED["batch_size"] <= 0 or FIXED["epochs"] <= 0: errors.append("batch_size and epochs must be positive")
    if FIXED["k_eval"] <= 0 or FIXED["eval_k_chunk_size"] <= 0: errors.append("evaluation K values must be positive")
    if FIXED["bootstrap_repetitions"] <= 0: errors.append("bootstrap_repetitions must be positive")
    if FIXED["saturation_relative_gain_fraction"] <= 0: errors.append("saturation gain fraction must be positive")
    if RUN_CONTROL["num_shards"] <= 0: errors.append("num_shards must be positive")
    if not (0 <= RUN_CONTROL["shard_index"] < RUN_CONTROL["num_shards"]): errors.append("shard_index outside [0,num_shards)")
    if len(EXPERIMENTAL["seeds"]) < 3:
        warnings.append("fewer than 3 seeds: final frozen-protocol bootstrap interpretation will be marked nonconforming")
    if set(EXPERIMENTAL["observation_resolutions"]) != {2,16,256}:
        warnings.append("R grid differs from the frozen main design")
    if set(EXPERIMENTAL["k_train_values"]) != {1,5,10,20,50}:
        warnings.append("K grid differs from the frozen main design")
    if set(EXPERIMENTAL["estimators_k_gt_1"]) != {"iwae","dreg"}:
        warnings.append("estimator grid differs from the frozen main design")
    if errors:
        raise ValueError("Configuration validation failed:\n- " + "\n- ".join(errors))
    return warnings

CONFIG_WARNINGS = validate_configuration()
for w in CONFIG_WARNINGS: print("CONFIG WARNING:", w)

ENVIRONMENT = environment_snapshot()
SOFTWARE_FINGERPRINT = sha256_json({
    "python": ENVIRONMENT["python_version_short"],
    "torch": ENVIRONMENT["torch"],
    "torchvision": ENVIRONMENT["torchvision"],
    "cuda_runtime": ENVIRONMENT.get("cuda_runtime"),
    "cudnn_version": ENVIRONMENT.get("cudnn_version"),
    "numpy": ENVIRONMENT["numpy"],
})
BENCHMARK_ENVIRONMENT_FINGERPRINT = sha256_json({
    "software": SOFTWARE_FINGERPRINT,
    "gpu_name": ENVIRONMENT.get("gpu_name"),
    "gpu_compute_capability": ENVIRONMENT.get("gpu_compute_capability"),
    "nvidia_driver": ENVIRONMENT.get("nvidia_driver"),
    "platform": ENVIRONMENT.get("platform"),
    "processor": ENVIRONMENT.get("processor"),
    "cpu_logical_count": ENVIRONMENT.get("cpu_logical_count"),
    "cpu_physical_count": ENVIRONMENT.get("cpu_physical_count"),
    "ram_total_gb": round(float(ENVIRONMENT.get("ram_total_gb",0.0)),3),
    "dtype": FIXED["dtype"],
    "adam_foreach": FIXED["adam_foreach"],
    "adam_fused": FIXED["adam_fused"],
    "adam_decoupled_weight_decay": FIXED["adam_decoupled_weight_decay"],
    "num_workers": FIXED["num_workers"],
    "pin_memory": FIXED["pin_memory"],
})

OUTPUT_ROOT = Path(RUN_CONTROL["output_root"]) / RUN_CONTROL["run_tag"]
OUTPUT_ROOT.mkdir(parents=True, exist_ok=True)
Path(RUN_CONTROL["data_root"]).mkdir(parents=True, exist_ok=True)
SESSION_ID = datetime.now(timezone.utc).strftime("%Y%m%dT%H%M%S_%fZ")
EXPORT_ROOT = OUTPUT_ROOT / "sessions" / SESSION_ID
EXPORT_ROOT.mkdir(parents=True, exist_ok=False)

set_global_seed(FIXED["split_seed"])

display(pd.DataFrame([ENVIRONMENT]).T.rename(columns={0:"value"}))
print("Software fingerprint:", SOFTWARE_FINGERPRINT)
print("Benchmark environment fingerprint:", BENCHMARK_ENVIRONMENT_FINGERPRINT)
print("Session export root:", EXPORT_ROOT)

,value
timestamp_utc,2026-09-08T05:23:17.727866+00:00
python,"3.12.3 (main, Jun 19 2026, 12:46:00) [GCC 13.3.0]"
python_version_short,3.12.3
platform,Linux-6.18.33.2-microsoft-standard-WSL2-x86_64...
processor,x86_64
cpu_logical_count,28
cpu_physical_count,14
ram_total_gb,31.223301
torch,2.11.0+cu130
torchvision,0.26.0+cu130


Software fingerprint: 502123363b288ea7619c9a3c2df64f5739efa2290b9453b9c0fc771ed8798265
Benchmark environment fingerprint: 13148c2274127eb438025f73524c662fbdd4df6f10fd235ea94166d36859e3ca
Session export root: /home/imohammedali/projects/pgm/latent_density_results/research_grade_z20_v2/sessions/20260908T052317_784076Z


## Fixed-hyperparameter justification

Exact values explicitly specified by the research plan are preserved unless the pilot gate rejects the current frozen protocol. Where the literature motivates a method but does **not** establish a unique optimal numeric value (for example batch size 128 or 100 epochs), the notebook says so instead of fabricating a citation. The scientific reason for holding such values fixed is to prevent nuisance-parameter changes from confounding `R`, `K`, or estimator comparisons.

In [ ]:
FIXED_RATIONALE = pd.DataFrame([
    ["input_dim", FIXED["input_dim"], "Native 28×28 image dimensionality; resizing would alter the representation and architecture.", "LeCun et al. 1998; Cohen et al. 2017; frozen plan"],
    ["hidden_dim", FIXED["hidden_dim"], "Two 200-unit deterministic tanh layers match the fixed study trunk and the DReG MNIST experimental precedent.", "Tucker et al., ICLR 2019; frozen plan"],
    ["latent_dim", FIXED["latent_dim"], "Hard study constraint. Held at 20 so latent capacity cannot become an additional factor; no claim that 20 is universally optimal.", "Frozen plan + current study constraint; Kingma & Welling 2014 for Gaussian latent VAE framework"],
    ["activation", FIXED["activation"], "Tanh is part of the frozen MLP trunk and appears in the IWAE/DReG MNIST family; changing it would introduce an architecture factor.", "Burda et al. 2016; Tucker et al. 2019; frozen plan"],
    ["prior/posterior", f"{FIXED['prior']} / {FIXED['posterior']}", "Standard-normal prior with diagonal-Gaussian amortized posterior is the canonical reparameterized VAE setup.", "Kingma & Welling, ICLR 2014"],
    ["weight_init", FIXED["weight_init"], "Variance-aware Glorot initialization is the specified heuristic for this tanh MLP.", "Glorot & Bengio, AISTATS 2010; Burda et al. 2016"],
    ["optimizer", FIXED["optimizer"], "Adam is appropriate for stochastic variational objectives and is the optimizer frozen by the plan.", "Kingma & Ba, ICLR 2015; Burda et al. 2016"],
    ["Adam β1,β2,ε", f"{FIXED['adam_betas']}, eps={FIXED['adam_eps']}", "Preserves the exact Adam hyperparameters specified from the IWAE protocol.", "Burda et al., ICLR 2016; frozen plan"],
    ["weight_decay", FIXED["weight_decay"], "No additional regularizer is introduced because regularization is not an experimental factor; zero also matches Adam's unregularized default form.", "Kingma & Ba 2015; PyTorch Adam docs; controlled-study rationale"],
    ["batch_size", FIXED["batch_size"], "Frozen compute choice from the plan. The exact value is not claimed literature-optimal; it is accepted only if the predeclared K=50 pilot stability gate passes.", "Frozen plan §7/§11; Burda et al. 2016 for minibatch IWAE training"],
    ["learning_rate", FIXED["lr"], "1e-3 is the top rate in Burda et al.'s schedule and the frozen starting rate; the pilot gate decides only whether this already-declared protocol is acceptable.", "Burda et al., ICLR 2016; frozen plan"],
    ["lr_schedule", FIXED["lr_schedule"], "Constant LR is the frozen default; a staged 50%/75% ×0.1 schedule is implemented only as the plan's predeclared alternative if a future pilot rerun explicitly freezes it.", "Frozen plan §7/§11"],
    ["epochs", FIXED["epochs"], "Compute-constrained frozen budget. The exact number is not literature-optimal and must pass the predeclared late-convergence pilot gate.", "Frozen plan §7/§11"],
    ["validation_fraction", FIXED["validation_fraction"], "Creates 10,000 MNIST validation examples and 18,800 EMNIST-Balanced validation examples from training only; fixed indices are reused across all conditions.", "Frozen plan split discipline; Cohen et al. 2017 dataset sizes"],
    ["k_eval", FIXED["k_eval"], "A common large evaluation K prevents training-K-dependent bound tightness from confounding final comparisons; 1000 is the plan's compute-scaled analogue of IWAE's large-K evaluation.", "Burda et al. 2016; frozen plan §6"],
    ["active_unit_threshold", FIXED["active_unit_threshold"], "Burda et al.'s activity statistic defines a latent unit as active above 1e-2.", "Burda et al., ICLR 2016"],
    ["R=2 likelihood", "Bernoulli", "Matches the binary VAE/IWAE observation model specified for the two-level representation.", "Kingma & Welling 2014; Burda et al. 2016; frozen plan"],
    ["R=16 likelihood", "Categorical(16)", "Exact probability mass over the 16 discrete levels; avoids using a continuous surrogate for a discrete target.", "Frozen plan; Theis et al., ICLR 2016 evaluation principles"],
    ["R=256 likelihood", "single discretized logistic", "Models exact discrete bin mass while preserving ordinal pixel structure without a 256-way head.", "Salimans et al., PixelCNN++, ICLR 2017; frozen plan"],
    ["min_logistic_scale", FIXED["min_logistic_scale"], "The exp(-7) scale floor follows the widely used PixelCNN++ numerical convention and protects CDF-difference calculations from collapse.", "Official OpenAI PixelCNN++ implementation; Salimans et al. 2017"],
    ["dtype", FIXED["dtype"], "Full FP32 avoids mixed-precision as an uninvestigated numerical factor and keeps all configurations on the same arithmetic protocol.", "PyTorch floating-point / reproducibility documentation; controlled-study rationale"],
    ["determinism", FIXED["deterministic_algorithms"], "Seeds and deterministic kernels reduce avoidable run-to-run variability; exact cross-platform bitwise identity is not promised.", "PyTorch official Reproducibility documentation"],
    ["bootstrap", f"{FIXED['bootstrap_repetitions']} reps", "Paired bootstrap is pre-specified because n=3/cell cannot support stable factorial-ANOVA assumptions; repetitions are fixed before results.", "Frozen statistical plan; Efron & Tibshirani, 1993"],
], columns=["parameter","fixed_value","reason_for_fixing","supporting_source"])

display(FIXED_RATIONALE)
FIXED_RATIONALE.to_csv(EXPORT_ROOT / "fixed_hyperparameter_rationale.csv", index=False)

ANALYSIS_CONVENTIONS = pd.DataFrame([
    ["saturation_relative_gain_fraction", FIXED["saturation_relative_gain_fraction"], "Operational reporting rule needed because the supplied plan requests a saturation point but supplies no numerical definition. It is descriptive, declared before results, and never used for training/model selection."],
    ["pilot late-improvement tolerance", PILOT["max_late_improvement_bpd"], "Predeclared fail-closed operational gate; not claimed as a literature-optimal hyperparameter."],
    ["pilot late-regression tolerance", PILOT["max_late_regression_bpd"], "Predeclared fail-closed operational gate; not claimed as a literature-optimal hyperparameter."],
    ["pilot gradient-spike ratio", PILOT["max_gradient_spike_ratio"], "Predeclared numerical-stability guard; not a model hyperparameter and never optimized against outcomes."],
], columns=["analysis_or_gate_control","fixed_value","interpretation"])
display(ANALYSIS_CONVENTIONS)
ANALYSIS_CONVENTIONS.to_csv(EXPORT_ROOT / "analysis_and_pilot_conventions.csv", index=False)# Every entry in FIXED receives an explicit audit rationale. Some numeric implementation controls
# (e.g., an RNG seed) cannot honestly be "derived" from a paper; in those cases the source supports
# the need to freeze/log the control, while the exact numeric value is explicitly labeled predeclared.
_FIXED_AUDIT = {
    "input_dim": ("Native 28×28 dimensionality; resizing would add a representation factor.", "LeCun et al. 1998; Cohen et al. 2017"),
    "hidden_dim": ("Frozen 200-unit MLP width; changing width changes model capacity.", "Tucker et al., ICLR 2019; frozen plan"),
    "latent_dim": ("Hard study constraint; never varied.", "Frozen plan + current user constraint; Kingma & Welling 2014 framework"),
    "activation": ("Frozen tanh architecture; changing activation creates an architecture factor.", "Burda et al. 2016; Tucker et al. 2019"),
    "prior": ("Canonical standard-normal VAE prior.", "Kingma & Welling, ICLR 2014"),
    "posterior": ("Diagonal Gaussian supports the reparameterization used by VAE/IWAE/DReG.", "Kingma & Welling 2014; Burda et al. 2016; Tucker et al. 2019"),
    "weight_init": ("Frozen variance-aware initialization for tanh networks.", "Glorot & Bengio, AISTATS 2010; frozen plan"),
    "dtype": ("FP32 is frozen to avoid mixed-precision as a hidden numerical/benchmark factor.", "PyTorch reproducibility/floating-point documentation; controlled-study rationale"),
    "optimizer": ("Optimizer is not a factor; Adam is frozen by the protocol.", "Kingma & Ba 2015; frozen plan"),
    "lr": ("Frozen starting learning rate from the plan; accepted only through the pilot gate.", "Burda et al. 2016; frozen plan"),
    "adam_betas": ("Exact optimizer moments specified by the frozen IWAE-derived protocol.", "Burda et al., ICLR 2016"),
    "adam_eps": ("Exact Adam epsilon specified by the frozen IWAE-derived protocol.", "Burda et al., ICLR 2016"),
    "weight_decay": ("No unplanned regularizer is introduced.", "Kingma & Ba 2015; controlled-study rationale"),
    "adam_amsgrad": ("Pins the optimizer algorithm to ordinary Adam rather than AMSGrad.", "PyTorch Adam documentation"),
    "adam_foreach": ("Pins implementation backend because PyTorch may otherwise choose foreach automatically on CUDA, changing memory/timing.", "PyTorch Adam documentation"),
    "adam_fused": ("Pins implementation backend; fused Adam is not silently introduced.", "PyTorch Adam documentation"),
    "adam_maximize": ("Standard minimization semantics are fixed.", "PyTorch Adam documentation"),
    "adam_capturable": ("CUDA-graph capture is not an experimental factor and is disabled.", "PyTorch Adam documentation"),
    "adam_differentiable": ("Optimizer-step autograd is not part of this estimator study.", "PyTorch Adam documentation"),
    "adam_decoupled_weight_decay": ("Pins ordinary Adam rather than silently switching to decoupled-weight-decay/AdamW semantics.", "PyTorch Adam documentation"),
    "epochs": ("Compute-constrained frozen budget; exact value is protocol-defined and must pass pilot late-convergence checks.", "Frozen plan §7/§11"),
    "batch_size": ("Protocol-defined compute choice; exact 128 is not claimed universally optimal and must pass K=50 pilot stability.", "Frozen plan §7/§11"),
    "lr_schedule": ("Constant is the frozen default; the staged alternative is only a predeclared future protocol revision after a failed pilot.", "Frozen plan §7/§11"),
    "gradient_clip_norm": ("No gradient clipping is silently added; clipping would change optimization dynamics.", "Controlled-study rationale; frozen plan does not specify clipping"),
    "validation_fraction": ("Fixed train-only holdout reused across all conditions; no test examples enter selection.", "Frozen plan split discipline; dataset sizes from LeCun/Cohen"),
    "split_seed": ("Exact seed is arbitrary but predeclared; fixing/logging it makes the partition reproducible and common to all conditions.", "PyTorch reproducibility guidance"),
    "k_eval": ("Common large evaluation K decouples evaluation-bound tightness from K_train.", "Burda et al. 2016; frozen plan §6"),
    "eval_k_chunk_size": ("Memory-only chunk size; exact log-sum-exp over all K is preserved, so this is not a scientific factor.", "Numerically stable log-sum-exp identity; PyTorch logsumexp documentation"),
    "active_unit_threshold": ("Uses the predeclared latent-activity diagnostic threshold.", "Burda et al., ICLR 2016"),
    "min_logistic_scale": ("Numerical floor follows the PixelCNN++ log-scale >= -7 convention.", "Salimans et al. 2017; official OpenAI PixelCNN++ implementation"),
    "bootstrap_repetitions": ("Monte-Carlo precision control for predeclared bootstrap summaries; fixed before seeing results.", "Efron & Tibshirani 1993; frozen statistical plan"),
    "bootstrap_seed": ("Exact seed is arbitrary but fixed so reported bootstrap intervals are reproducible.", "Bootstrap reproducibility rationale; PyTorch/NumPy RNG guidance"),
    "saturation_relative_gain_fraction": ("Predeclared descriptive operational rule because the source requests a saturation point but gives no numeric criterion.", "Study analysis convention; not claimed literature-optimal"),
    "num_workers": ("Input-pipeline worker count is fixed so systems overhead is not changed between benchmarked conditions.", "PyTorch DataLoader documentation; controlled-benchmark rationale"),
    "pin_memory": ("Pinned-memory behavior is held constant across runs.", "PyTorch DataLoader documentation"),
    "persistent_workers": ("Worker lifecycle is fixed across runs to avoid a systems-level benchmark confound.", "PyTorch DataLoader documentation"),
    "deterministic_algorithms": ("Requests deterministic kernels to reduce avoidable implementation variance.", "PyTorch Reproducibility documentation"),
    "allow_tf32": ("TF32 is disabled so arithmetic mode does not differ silently by GPU generation.", "PyTorch CUDA numerical-accuracy documentation"),
    "benchmark_batch_size": ("Benchmark load is fixed across compared models; exact value is a systems measurement convention, not a tuned scientific parameter.", "PyTorch benchmarking guidance"),
    "benchmark_warmup": ("Warm-up avoids first-call/lazy-initialization timing bias.", "PyTorch benchmarking guidance"),
    "benchmark_repetitions": ("Repeated timing reduces single-measurement noise; count fixed before outcomes.", "PyTorch benchmarking guidance"),
    "ram_poll_interval_sec": ("RAM sampling cadence is fixed and reported as an approximation, not an exact event-level maximum.", "psutil/process-monitoring methodology; controlled-benchmark rationale"),
}
missing_fixed=set(FIXED)-set(_FIXED_AUDIT)
extra_fixed=set(_FIXED_AUDIT)-set(FIXED)
assert not missing_fixed and not extra_fixed, (missing_fixed,extra_fixed)
FIXED_CONTROL_AUDIT=pd.DataFrame([
    {"parameter":k,"fixed_value":FIXED[k],"reason_for_fixing":_FIXED_AUDIT[k][0],"supporting_source":_FIXED_AUDIT[k][1]}
    for k in FIXED
])
display(FIXED_CONTROL_AUDIT)
FIXED_CONTROL_AUDIT.to_csv(EXPORT_ROOT / "fixed_control_complete_audit.csv",index=False)

,parameter,fixed_value,reason_for_fixing,supporting_source
0,input_dim,784,Native 28×28 image dimensionality; resizing wo...,LeCun et al. 1998; Cohen et al. 2017; frozen plan
1,hidden_dim,200,Two 200-unit deterministic tanh layers match t...,"Tucker et al., ICLR 2019; frozen plan"
2,latent_dim,20,Hard study constraint. Held at 20 so latent ca...,Frozen plan + current study constraint; Kingma...
3,activation,tanh,Tanh is part of the frozen MLP trunk and appea...,Burda et al. 2016; Tucker et al. 2019; frozen ...
4,prior/posterior,standard_normal / diagonal_gaussian,Standard-normal prior with diagonal-Gaussian a...,"Kingma & Welling, ICLR 2014"
5,weight_init,xavier_uniform,Variance-aware Glorot initialization is the sp...,"Glorot & Bengio, AISTATS 2010; Burda et al. 2016"
6,optimizer,adam,Adam is appropriate for stochastic variational...,"Kingma & Ba, ICLR 2015; Burda et al. 2016"
7,"Adam β1,β2,ε","(0.9, 0.999), eps=0.0001",Preserves the exact Adam hyperparameters speci...,"Burda et al., ICLR 2016; frozen plan"
8,weight_decay,0.0,No additional regularizer is introduced becaus...,Kingma & Ba 2015; PyTorch Adam docs; controlle...
9,batch_size,128,Frozen compute choice from the plan. The exact...,Frozen plan §7/§11; Burda et al. 2016 for mini...


,analysis_or_gate_control,fixed_value,interpretation
0,saturation_relative_gain_fraction,0.100,Operational reporting rule needed because the ...
1,pilot late-improvement tolerance,0.005,Predeclared fail-closed operational gate; not ...
2,pilot late-regression tolerance,0.010,Predeclared fail-closed operational gate; not ...
3,pilot gradient-spike ratio,100.000,Predeclared numerical-stability guard; not a m...


,parameter,fixed_value,reason_for_fixing,supporting_source
0,input_dim,784,Native 28×28 dimensionality; resizing would ad...,LeCun et al. 1998; Cohen et al. 2017
1,hidden_dim,200,Frozen 200-unit MLP width; changing width chan...,"Tucker et al., ICLR 2019; frozen plan"
2,latent_dim,20,Hard study constraint; never varied.,Frozen plan + current user constraint; Kingma ...
3,activation,tanh,Frozen tanh architecture; changing activation ...,Burda et al. 2016; Tucker et al. 2019
4,prior,standard_normal,Canonical standard-normal VAE prior.,"Kingma & Welling, ICLR 2014"
5,posterior,diagonal_gaussian,Diagonal Gaussian supports the reparameterizat...,Kingma & Welling 2014; Burda et al. 2016; Tuck...
6,weight_init,xavier_uniform,Frozen variance-aware initialization for tanh ...,"Glorot & Bengio, AISTATS 2010; frozen plan"
7,dtype,float32,FP32 is frozen to avoid mixed-precision as a h...,PyTorch reproducibility/floating-point documen...
8,optimizer,adam,Optimizer is not a factor; Adam is frozen by t...,Kingma & Ba 2015; frozen plan
9,lr,0.001,Frozen starting learning rate from the plan; a...,Burda et al. 2016; frozen plan


# 5. Dataset Loading & Inspection

The official test split remains sealed from tuning/model selection. Validation indices are generated once from each official training split using a fixed split seed and are reused across every resolution, K, estimator, and model seed.

To make future audits possible, the notebook hashes the canonical training/test image tensors, labels, and split indices. Hashing test bytes is provenance recording only; no test statistic is used to choose a model or protocol.

In [ ]:
@dataclass
class DatasetBundle:
    name: str
    split_name: str
    train_images: torch.Tensor
    train_labels: torch.Tensor
    val_images: torch.Tensor
    val_labels: torch.Tensor
    test_images: torch.Tensor
    test_labels: torch.Tensor
    train_indices: torch.Tensor
    val_indices: torch.Tensor
    dataset_fingerprint: str
    provenance: dict


def sha256_tensor(t: torch.Tensor):
    a = t.detach().cpu().contiguous().numpy()
    return hashlib.sha256(memoryview(a)).hexdigest()


def fixed_train_val_indices(n: int):
    g = torch.Generator().manual_seed(int(FIXED["split_seed"]))
    perm = torch.randperm(n, generator=g)
    n_val = int(round(n * float(FIXED["validation_fraction"])))
    if not (0 < n_val < n):
        raise ValueError("Invalid validation size")
    val_idx = perm[:n_val].contiguous()
    train_idx = perm[n_val:].contiguous()
    # tensor-level set logic without constructing huge Python sets
    assert torch.unique(torch.cat([train_idx, val_idx])).numel() == n
    return train_idx, val_idx


def load_dataset_bundle(name: str):
    root = RUN_CONTROL["data_root"]
    download = bool(RUN_CONTROL["download_data"])
    if name == "mnist":
        split_name = "official"
        full_train = datasets.MNIST(root=root, train=True, download=download)
        test = datasets.MNIST(root=root, train=False, download=download)
    elif name == "emnist":
        split_name = EMNIST_CONFIG["split"]
        full_train = datasets.EMNIST(root=root, split=split_name, train=True, download=download)
        test = datasets.EMNIST(root=root, split=split_name, train=False, download=download)
    else:
        raise ValueError(name)

    train_idx, val_idx = fixed_train_val_indices(len(full_train))
    data = full_train.data.clone().contiguous()
    targets = full_train.targets.clone().contiguous()
    test_data = test.data.clone().contiguous()
    test_targets = test.targets.clone().contiguous()

    prov = {
        "dataset": name,
        "split_name": split_name,
        "torchvision_version": torchvision.__version__,
        "official_train_n": len(data),
        "official_test_n": len(test_data),
        "train_images_sha256": sha256_tensor(data),
        "train_targets_sha256": sha256_tensor(targets),
        "test_images_sha256": sha256_tensor(test_data),
        "test_targets_sha256": sha256_tensor(test_targets),
        "train_indices_sha256": sha256_tensor(train_idx),
        "val_indices_sha256": sha256_tensor(val_idx),
        "split_seed": FIXED["split_seed"],
        "validation_fraction": FIXED["validation_fraction"],
    }
    fingerprint = sha256_json(prov)
    return DatasetBundle(
        name=name, split_name=split_name,
        train_images=data[train_idx], train_labels=targets[train_idx],
        val_images=data[val_idx], val_labels=targets[val_idx],
        test_images=test_data, test_labels=test_targets,
        train_indices=train_idx, val_indices=val_idx,
        dataset_fingerprint=fingerprint, provenance=prov,
    )

DATASETS = {}
if PHASES["run_mnist_main_grid"] or PILOT["enabled"] or PHASES["run_emnist_generalization_grid"]:
    DATASETS["mnist"] = load_dataset_bundle("mnist")
# EMNIST is deliberately NOT loaded or inspected here. Its dataset is opened only after
# the complete MNIST Phase-13 gate, preserving the Phase-16 sequencing in the frozen plan.

SPLIT_DIR = EXPORT_ROOT / "dataset_splits"
SPLIT_DIR.mkdir(parents=True, exist_ok=True)
rows=[]
for key,b in DATASETS.items():
    np.save(SPLIT_DIR / f"{key}_{b.split_name}_train_indices.npy", b.train_indices.numpy())
    np.save(SPLIT_DIR / f"{key}_{b.split_name}_val_indices.npy", b.val_indices.numpy())
    with open(SPLIT_DIR / f"{key}_{b.split_name}_provenance.json", "w") as f:
        json.dump({**b.provenance, "dataset_fingerprint": b.dataset_fingerprint}, f, indent=2)
    train_f = b.train_images.float().div(255.0)
    val_f = b.val_images.float().div(255.0)
    rows.append({
        "dataset":key, "split":b.split_name,
        "train_n":len(b.train_images), "val_n":len(b.val_images), "test_n_sealed":len(b.test_images),
        "train_pixel_mean":float(train_f.mean()), "train_pixel_std":float(train_f.std()),
        "val_pixel_mean":float(val_f.mean()), "val_pixel_std":float(val_f.std()),
        "train_classes":int(torch.unique(b.train_labels).numel()),
        "dataset_fingerprint":b.dataset_fingerprint,
    })
DATASET_STATS = pd.DataFrame(rows)
display(DATASET_STATS)
DATASET_STATS.to_csv(EXPORT_ROOT / "dataset_statistics_pretest.csv", index=False)

,dataset,split,train_n,val_n,test_n_sealed,train_pixel_mean,train_pixel_std,val_pixel_mean,val_pixel_std,train_classes,dataset_fingerprint
0,mnist,official,50000,10000,10000,0.13065,0.308091,0.130714,0.308192,10,fc0ca319e49b1ead75e8048ed6b4a97e29cafe153077f1...


# 6. Preprocessing

Quantization is deterministic and uses the same original byte tensor for every `R`:

\[
q_R = \operatorname{round}\left(\frac{u}{255}(R-1)\right),\qquad x_R=\frac{q_R}{R-1}.
\]

For `R=256`, `q=u` exactly. Labels are carried only for inspection; the density model is unsupervised.

In [ ]:
class QuantizedTensorDataset(Dataset):
    def __init__(self, images_u8, labels, resolution):
        if images_u8.dtype != torch.uint8:
            raise TypeError("Expected uint8 source images")
        if int(resolution) not in {2,16,256}:
            raise ValueError(resolution)
        self.images = images_u8
        self.labels = labels
        self.resolution = int(resolution)
    def __len__(self): return len(self.images)
    def __getitem__(self, idx):
        u = self.images[idx]
        if self.resolution == 256:
            q = u.long()
        else:
            q = torch.round(u.float() * ((self.resolution - 1) / 255.0)).long()
        x = q.float() / float(self.resolution - 1)
        return x.reshape(-1), q.reshape(-1), self.labels[idx]


def seed_worker(worker_id):
    worker_seed = torch.initial_seed() % (2**32)
    np.random.seed(worker_seed)
    random.seed(worker_seed)


def make_loader(images, labels, resolution, batch_size, shuffle, loader_seed):
    ds = QuantizedTensorDataset(images, labels, resolution)
    g = torch.Generator().manual_seed(int(loader_seed))
    return DataLoader(
        ds,
        batch_size=int(batch_size),
        shuffle=bool(shuffle),
        num_workers=int(FIXED["num_workers"]),
        pin_memory=bool(FIXED["pin_memory"] and torch.cuda.is_available()),
        persistent_workers=bool(FIXED["persistent_workers"] and FIXED["num_workers"] > 0),
        worker_init_fn=seed_worker if FIXED["num_workers"] > 0 else None,
        generator=g,
        drop_last=False,
    )


def quantization_summary(bundle, max_images=5000):
    images = bundle.train_images[:max_images]
    out=[]
    for r in EXPERIMENTAL["observation_resolutions"]:
        q = images.long() if r==256 else torch.round(images.float()*((r-1)/255.0)).long()
        out.append({
            "dataset":bundle.name, "split":bundle.split_name, "R":int(r),
            "observed_levels":int(torch.unique(q).numel()),
            "min_level":int(q.min()), "max_level":int(q.max()),
            "mean_normalized":float((q.float()/(r-1)).mean()),
        })
    return out

QROWS=[]
for b in DATASETS.values(): QROWS.extend(quantization_summary(b))
QUANTIZATION_STATS=pd.DataFrame(QROWS)
display(QUANTIZATION_STATS)
QUANTIZATION_STATS.to_csv(EXPORT_ROOT / "quantization_statistics.csv", index=False)

,dataset,split,R,observed_levels,min_level,max_level,mean_normalized
0,mnist,official,2,2,0,1,0.132611
1,mnist,official,16,16,0,15,0.131552
2,mnist,official,256,256,0,255,0.131004


# 7. Model Definition

The common trunk and latent stochastic layer are identical across resolution conditions. The code explicitly refuses unsupported activation/prior/posterior/initialization choices so the saved configuration can never claim one method while executing another.

In [ ]:
class LatentDensityMLP(nn.Module):
    def __init__(self, resolution):
        super().__init__()
        if FIXED["latent_dim"] != 20: raise AssertionError("latent_dim must be 20")
        if FIXED["activation"] != "tanh": raise ValueError("Only frozen tanh activation is supported")
        if FIXED["prior"] != "standard_normal" or FIXED["posterior"] != "diagonal_gaussian":
            raise ValueError("Only the frozen Gaussian latent model is supported")
        if FIXED["weight_init"] != "xavier_uniform": raise ValueError("Only xavier_uniform is supported")
        self.resolution=int(resolution)
        d,h,z = FIXED["input_dim"], FIXED["hidden_dim"], FIXED["latent_dim"]
        self.enc1=nn.Linear(d,h); self.enc2=nn.Linear(h,h)
        self.enc_mu=nn.Linear(h,z); self.enc_logstd=nn.Linear(h,z)
        self.dec1=nn.Linear(z,h); self.dec2=nn.Linear(h,h)
        outdim={2:d,16:d*16,256:d*2}[self.resolution]
        self.head=nn.Linear(h,outdim)
        self.reset_parameters()

    def reset_parameters(self):
        for m in self.modules():
            if isinstance(m,nn.Linear):
                nn.init.xavier_uniform_(m.weight)
                nn.init.zeros_(m.bias)

    def encode(self,x):
        h=torch.tanh(self.enc1(x)); h=torch.tanh(self.enc2(h))
        return self.enc_mu(h), self.enc_logstd(h)

    def reparameterize(self,mu,log_std,k,generator=None):
        eps=torch.randn((int(k),)+tuple(mu.shape),device=mu.device,dtype=mu.dtype,generator=generator)
        return mu.unsqueeze(0)+torch.exp(log_std).unsqueeze(0)*eps

    def decode(self,z):
        h=torch.tanh(self.dec1(z)); h=torch.tanh(self.dec2(h)); raw=self.head(h)
        if self.resolution==2:
            return {"bernoulli_logits":raw}
        if self.resolution==16:
            return {"categorical_logits":raw.view(*raw.shape[:-1],FIXED["input_dim"],16)}
        raw=raw.view(*raw.shape[:-1],FIXED["input_dim"],2)
        loc=torch.tanh(raw[...,0])
        scale=torch.clamp(F.softplus(raw[...,1]),min=float(FIXED["min_logistic_scale"]))
        return {"logistic_loc":loc,"logistic_scale":scale}

    def inference_parameters(self):
        mods=[self.enc1,self.enc2,self.enc_mu,self.enc_logstd]
        return [p for m in mods for p in m.parameters()]
    def generative_parameters(self):
        mods=[self.dec1,self.dec2,self.head]
        return [p for m in mods for p in m.parameters()]


def parameter_metrics(model):
    total=sum(p.numel() for p in model.parameters())
    trainable=sum(p.numel() for p in model.parameters() if p.requires_grad)
    bytes_=sum(p.numel()*p.element_size() for p in model.parameters())
    return total,trainable,bytes_


def linear_macs_per_example(resolution,k_train):
    d,h,z=FIXED["input_dim"],FIXED["hidden_dim"],FIXED["latent_dim"]
    outdim={2:d,16:d*16,256:d*2}[int(resolution)]
    encoder = d*h + h*h + h*z + h*z
    decoder_per_sample = z*h + h*h + h*outdim
    total = encoder + int(k_train)*decoder_per_sample
    return {
        "encoder_linear_macs_per_example":int(encoder),
        "decoder_linear_macs_per_latent_sample":int(decoder_per_sample),
        "forward_linear_macs_per_example":int(total),
        "forward_linear_flops_per_example_approx":int(2*total),
    }

ARCH_ROWS=[]
for r in [2,16,256]:
    set_global_seed(0)
    m=LatentDensityMLP(r)
    total,trainable,bytes_=parameter_metrics(m)
    ARCH_ROWS.append({"R":r,"parameters":total,"trainable_parameters":trainable,"parameter_mb":bytes_/(1024**2),**linear_macs_per_example(r,1)})
ARCHITECTURE_COST=pd.DataFrame(ARCH_ROWS)
display(ARCHITECTURE_COST)
ARCHITECTURE_COST.to_csv(EXPORT_ROOT / "architecture_cost_by_resolution.csv",index=False)

,R,parameters,trainable_parameters,parameter_mb,encoder_linear_macs_per_example,decoder_linear_macs_per_latent_sample,forward_linear_macs_per_example,forward_linear_flops_per_example_approx
0,2,407224,407224,1.553436,204800,200800,405600,811200
1,16,2770984,2770984,10.570465,204800,2552800,2757600,5515200
2,256,564808,564808,2.154572,204800,357600,562400,1124800


# 8. Training Pipeline

`K=1` is trained as the VAE baseline using a single reparameterized reconstruction sample plus the **analytic Gaussian KL**, matching the mathematical VAE objective in the plan. `K>1` uses the standard IWAE bound. DReG uses the same forward samples and importance weights as IWAE; only the inference-network gradient is replaced by the doubly reparameterized estimator, while the generative-network gradient remains the ordinary IWAE gradient.

In [ ]:
LOG_2PI=math.log(2.0*math.pi)
LN2=math.log(2.0)
PIXELS=FIXED["input_dim"]


def diag_normal_log_prob(z,mu,log_std):
    return -0.5*(((z-mu)/torch.exp(log_std))**2 + 2.0*log_std + LOG_2PI).sum(-1)

def standard_normal_log_prob(z):
    return -0.5*(z**2+LOG_2PI).sum(-1)

def analytic_kl_standard_normal(mu,log_std):
    var=torch.exp(2.0*log_std)
    return 0.5*(mu.pow(2)+var-1.0-2.0*log_std).sum(-1)


def discretized_logistic_log_mass(q_idx,loc,scale):
    """Per-pixel log probability mass for 8-bit bins mapped to [-1,1]."""
    x=(q_idx.float()/255.0)*2.0-1.0
    x=x.unsqueeze(0).expand_as(loc)
    half_bin=1.0/255.0
    centered=x-loc
    plus_in=(centered+half_bin)/scale
    min_in=(centered-half_bin)/scale
    log_cdf_plus=F.logsigmoid(plus_in)
    log_one_minus_cdf_min=F.logsigmoid(-min_in)
    cdf_delta=torch.sigmoid(plus_in)-torch.sigmoid(min_in)
    mid_in=centered/scale
    log_pdf_mid=mid_in-torch.log(scale)-2.0*F.softplus(mid_in)
    log_mid=torch.where(cdf_delta>1e-5,torch.log(torch.clamp(cdf_delta,min=1e-12)),log_pdf_mid+math.log(2.0/255.0))
    q=q_idx.unsqueeze(0).expand_as(loc)
    return torch.where(q==0,log_cdf_plus,torch.where(q==255,log_one_minus_cdf_min,log_mid))


def discretized_logistic_log_prob(q_idx,loc,scale):
    return discretized_logistic_log_mass(q_idx,loc,scale).sum(-1)


def observation_log_prob(model,decoded,q_idx):
    if model.resolution==2:
        targets=q_idx.float().unsqueeze(0).expand_as(decoded["bernoulli_logits"])
        return -F.binary_cross_entropy_with_logits(decoded["bernoulli_logits"],targets,reduction="none").sum(-1)
    if model.resolution==16:
        logits=decoded["categorical_logits"]
        target=q_idx.unsqueeze(0).unsqueeze(-1).expand(*logits.shape[:-1],1)
        return torch.log_softmax(logits,-1).gather(-1,target).squeeze(-1).sum(-1)
    return discretized_logistic_log_prob(q_idx,decoded["logistic_loc"],decoded["logistic_scale"])


def compute_log_weight_parts(model,x,q_idx,k,generator=None):
    mu,log_std=model.encode(x)
    z=model.reparameterize(mu,log_std,k,generator=generator)
    decoded=model.decode(z)
    log_px=observation_log_prob(model,decoded,q_idx)
    log_pz=standard_normal_log_prob(z)
    log_qz=diag_normal_log_prob(z,mu.unsqueeze(0),log_std.unsqueeze(0))
    return {"mu":mu,"log_std":log_std,"z":z,"log_px":log_px,"log_pz":log_pz,"log_qz":log_qz,"log_w":log_px+log_pz-log_qz}


def negative_iwae_bound(log_w):
    k=log_w.shape[0]
    return -(torch.logsumexp(log_w,dim=0)-math.log(k)).mean()


def vae_training_parts(model,x,q_idx,generator=None):
    mu,log_std=model.encode(x)
    z=model.reparameterize(mu,log_std,1,generator=generator)
    log_px=observation_log_prob(model,model.decode(z),q_idx).squeeze(0)
    kl=analytic_kl_standard_normal(mu,log_std)
    loss=-(log_px-kl).mean()
    return {"loss":loss,"mu":mu,"log_std":log_std,"log_px":log_px.unsqueeze(0),"kl":kl}


def _assign_grads(params,grads):
    for p,g in zip(params,grads): p.grad=g


def _gradient_norm_tensor(model):
    terms=[]
    for p in model.parameters():
        if p.grad is not None: terms.append(p.grad.detach().float().pow(2).sum())
    if not terms: return torch.tensor(0.0,device=DEVICE)
    return torch.sqrt(torch.stack(terms).sum())


def train_step(model,optimizer,x,q_idx,k,method,collect_grad_diagnostics=False):
    optimizer.zero_grad(set_to_none=True)
    if method=="vae":
        if int(k)!=1: raise ValueError("VAE baseline is only K=1")
        v=vae_training_parts(model,x,q_idx)
        loss=v["loss"]
        loss.backward()
        log_px=v["log_px"]
        mu,log_std=v["mu"],v["log_std"]
    else:
        if int(k)<=1: raise ValueError("IWAE/DReG distinct estimators require K>1")
        parts=compute_log_weight_parts(model,x,q_idx,k)
        loss=negative_iwae_bound(parts["log_w"])
        log_px=parts["log_px"]; mu,log_std=parts["mu"],parts["log_std"]
        if method=="iwae":
            loss.backward()
        elif method=="dreg":
            theta=model.generative_parameters(); phi=model.inference_parameters()
            theta_grads=torch.autograd.grad(loss,theta,retain_graph=True,create_graph=False,allow_unused=False)
            # Stop the explicit q-parameter path, preserving only the pathwise z dependence.
            log_q_stl=diag_normal_log_prob(parts["z"],parts["mu"].detach().unsqueeze(0),parts["log_std"].detach().unsqueeze(0))
            log_w_stl=parts["log_px"]+parts["log_pz"]-log_q_stl
            normalized_w=torch.softmax(parts["log_w"].detach(),dim=0)
            phi_surrogate=-(normalized_w.square()*log_w_stl).sum(0).mean()
            phi_grads=torch.autograd.grad(phi_surrogate,phi,retain_graph=False,create_graph=False,allow_unused=False)
            _assign_grads(theta,theta_grads); _assign_grads(phi,phi_grads)
        else:
            raise ValueError(method)

    if FIXED["gradient_clip_norm"] is not None:
        torch.nn.utils.clip_grad_norm_(model.parameters(),float(FIXED["gradient_clip_norm"]))
    grad_norm=_gradient_norm_tensor(model) if collect_grad_diagnostics else None
    optimizer.step()

    recon_nll_bpd=(-log_px.mean())/(LN2*PIXELS)
    kl_bpd=analytic_kl_standard_normal(mu,log_std).mean()/(LN2*PIXELS)
    return {
        "loss_nats_per_image":loss.detach(),
        "objective_bpd":loss.detach()/(LN2*PIXELS),
        "recon_nll_bpd":recon_nll_bpd.detach(),
        "analytic_kl_bpd":kl_bpd.detach(),
        "grad_norm_l2":grad_norm,
    }

## Objective / likelihood sanity tests

These tests are deliberately independent of a successful training loop. They verify finite likelihoods for all three heads, K=1 IWAE algebra, DReG gradient construction, generative-gradient equality by construction, and normalization of the 256-bin discretized-logistic mass.

In [ ]:
def objective_sanity_tests():
    set_global_seed(12345)
    rows=[]
    for r in [2,16,256]:
        m=LatentDensityMLP(r).to(DEVICE)
        B=4
        q=torch.randint(0,r,(B,PIXELS),device=DEVICE)
        x=q.float()/float(r-1)
        parts=compute_log_weight_parts(m,x,q,5)
        assert torch.isfinite(parts["log_w"]).all()
        rows.append({"R":r,"finite_log_weights":True,"mean_log_weight":float(parts["log_w"].mean().detach().cpu())})

    # K=1 IWAE algebra: logmeanexp over one sample is exactly that sample's log-weight.
    m=LatentDensityMLP(2).to(DEVICE); q=torch.randint(0,2,(3,PIXELS),device=DEVICE); x=q.float()
    p1=compute_log_weight_parts(m,x,q,1)
    assert torch.allclose(negative_iwae_bound(p1["log_w"]),-p1["log_w"].mean(),atol=1e-7,rtol=1e-6)

    # DReG: independently compute standard theta gradient and DReG phi gradient; both must be finite.
    m=LatentDensityMLP(2).to(DEVICE); q=torch.randint(0,2,(3,PIXELS),device=DEVICE); x=q.float()
    parts=compute_log_weight_parts(m,x,q,5); loss=negative_iwae_bound(parts["log_w"])
    theta=m.generative_parameters(); phi=m.inference_parameters()
    theta_std=torch.autograd.grad(loss,theta,retain_graph=True)
    log_q_stl=diag_normal_log_prob(parts["z"],parts["mu"].detach().unsqueeze(0),parts["log_std"].detach().unsqueeze(0))
    w=torch.softmax(parts["log_w"].detach(),0)
    phi_sur=-(w.square()*(parts["log_px"]+parts["log_pz"]-log_q_stl)).sum(0).mean()
    phi_dreg=torch.autograd.grad(phi_sur,phi)
    assert all(torch.isfinite(g).all() for g in theta_std+phi_dreg)

    # Direct implementation regression test: with identical initialization/samples and zero learning rate,
    # train_step must assign the same generative-network gradient for IWAE and DReG.
    base=LatentDensityMLP(2).to(DEVICE); state={k:v.detach().clone() for k,v in base.state_dict().items()}
    mi=LatentDensityMLP(2).to(DEVICE); md=LatentDensityMLP(2).to(DEVICE); mi.load_state_dict(state); md.load_state_dict(state)
    oi=torch.optim.SGD(mi.parameters(),lr=0.0); od=torch.optim.SGD(md.parameters(),lr=0.0)
    q=torch.randint(0,2,(4,PIXELS),device=DEVICE); x=q.float()
    set_global_seed(777); train_step(mi,oi,x,q,5,"iwae",False); gi=[p.grad.detach().clone() for p in mi.generative_parameters()]
    set_global_seed(777); train_step(md,od,x,q,5,"dreg",False); gd=[p.grad.detach().clone() for p in md.generative_parameters()]
    assert all(torch.allclose(a,b,atol=2e-5,rtol=2e-4) for a,b in zip(gi,gd)), "DReG generative gradient diverged from IWAE"

    # Discretized logistic normalization for several locations/scales across all 256 bins.
    levels=torch.arange(256,device=DEVICE).view(256,1)
    norm_errors=[]
    for loc_value,scale_value in [(-0.7,0.05),(0.0,0.2),(0.8,1.0),(0.0,float(FIXED["min_logistic_scale"]))]:
        q_all=levels.long()
        loc=torch.full((1,256,1),loc_value,device=DEVICE)
        scale=torch.full_like(loc,scale_value)
        masses=torch.exp(discretized_logistic_log_mass(q_all,loc,scale).squeeze(-1).squeeze(0))
        err=abs(float(masses.sum().detach().cpu())-1.0)
        norm_errors.append(err)
    assert max(norm_errors)<5e-4, norm_errors
    return pd.DataFrame(rows),max(norm_errors)

SANITY_TABLE,MAX_LOGISTIC_NORM_ERROR=objective_sanity_tests()
display(SANITY_TABLE)
print("Max discretized-logistic normalization error:",MAX_LOGISTIC_NORM_ERROR)

,R,finite_log_weights,mean_log_weight
0,2,True,-558.084106
1,16,True,-2215.313721
2,256,True,-4749.196289


Max discretized-logistic normalization error: 2.0265579223632812e-06


# 9. Experimental Runner

Training and validation are deliberately separated for timing/memory accounting. Model initialization, batch order, and Monte-Carlo streams are paired across comparable methods. Validation checkpoint selection uses fixed Monte-Carlo random numbers across epochs. Completed experiments are immutable; retries create new attempts, and epoch checkpoints permit Kaggle-session recovery.

In [ ]:
class PeakRSSMonitor:
    def __init__(self, interval=0.2):
        self.interval=float(interval); self.peak=0; self._stop=threading.Event(); self._thread=None
    def _rss(self):
        p=psutil.Process(os.getpid()); total=p.memory_info().rss
        for c in p.children(recursive=True):
            try: total += c.memory_info().rss
            except (psutil.NoSuchProcess, psutil.AccessDenied): pass
        return total
    def _run(self):
        while not self._stop.is_set():
            self.peak=max(self.peak,self._rss()); self._stop.wait(self.interval)
    def __enter__(self):
        self.peak=self._rss(); self._thread=threading.Thread(target=self._run,daemon=True); self._thread.start(); return self
    def __exit__(self,*args):
        self._stop.set(); self._thread.join(); self.peak=max(self.peak,self._rss())


def cuda_sync():
    if torch.cuda.is_available(): torch.cuda.synchronize()


def new_torch_generator(seed):
    return torch.Generator(device=DEVICE).manual_seed(int(seed))


def make_optimizer(model):
    if FIXED["optimizer"]!="adam": raise ValueError("Only frozen Adam is supported")
    kwargs=dict(lr=float(FIXED["lr"]),betas=tuple(FIXED["adam_betas"]),eps=float(FIXED["adam_eps"]),
                weight_decay=float(FIXED["weight_decay"]),amsgrad=bool(FIXED["adam_amsgrad"]))
    sig=inspect.signature(torch.optim.Adam)
    optional={"foreach":FIXED["adam_foreach"],"fused":FIXED["adam_fused"],"maximize":FIXED["adam_maximize"],
              "capturable":FIXED["adam_capturable"],"differentiable":FIXED["adam_differentiable"],
              "decoupled_weight_decay":FIXED["adam_decoupled_weight_decay"]}
    for k,v in optional.items():
        if k in sig.parameters: kwargs[k]=v
    opt=torch.optim.Adam(model.parameters(),**kwargs)
    return opt,kwargs


def apply_lr_schedule(optimizer, epoch_1based):
    base=float(FIXED["lr"]); schedule=FIXED["lr_schedule"]
    if schedule=="constant": lr=base
    elif schedule=="staged_50_75_x0.1":
        frac=(epoch_1based-1)/max(1,FIXED["epochs"])
        lr=base*(0.1 if frac>=0.75 else (1.0 if frac<0.50 else 0.1))
        if frac>=0.75: lr=base*0.01
    else: raise ValueError(schedule)
    for g in optimizer.param_groups: g["lr"]=lr
    return lr


def _gpu_peak_reset():
    if torch.cuda.is_available():
        cuda_sync(); torch.cuda.reset_peak_memory_stats(); return torch.cuda.memory_allocated()
    return 0

def _gpu_peak_read(base=0):
    if not torch.cuda.is_available(): return {"peak_allocated_bytes":0,"peak_reserved_bytes":0,"peak_delta_allocated_bytes":0}
    cuda_sync()
    return {"peak_allocated_bytes":int(torch.cuda.max_memory_allocated()),
            "peak_reserved_bytes":int(torch.cuda.max_memory_reserved()),
            "peak_delta_allocated_bytes":int(max(0,torch.cuda.max_memory_allocated()-base))}


def train_one_epoch(model,optimizer,bundle,resolution,k_train,method,model_seed,epoch_1based,collect_grad_diagnostics=False):
    model.train(); lr=apply_lr_schedule(optimizer,epoch_1based)
    epoch_seed=stable_seed("train_epoch",bundle.name,bundle.split_name,resolution,model_seed,epoch_1based)
    set_global_seed(epoch_seed)
    loader=make_loader(bundle.train_images,bundle.train_labels,resolution,FIXED["batch_size"],True,
                       stable_seed("loader",bundle.name,resolution,model_seed,epoch_1based))
    totals=defaultdict(lambda: torch.tensor(0.0,device=DEVICE)); n=0; grad_values=[]
    base_gpu=_gpu_peak_reset(); cuda_sync(); t0=time.perf_counter()
    with PeakRSSMonitor(FIXED["ram_poll_interval_sec"]) as rss:
        for x,q,_ in loader:
            x=x.to(DEVICE,non_blocking=True); q=q.to(DEVICE,non_blocking=True)
            out=train_step(model,optimizer,x,q,k_train,method,collect_grad_diagnostics)
            bs=x.shape[0]; n+=bs
            for key in ["loss_nats_per_image","objective_bpd","recon_nll_bpd","analytic_kl_bpd"]:
                totals[key]+=out[key]*bs
            if collect_grad_diagnostics: grad_values.append(out["grad_norm_l2"].detach())
    cuda_sync(); elapsed=time.perf_counter()-t0; gpu=_gpu_peak_read(base_gpu)
    result={k:float((v/n).cpu()) for k,v in totals.items()}
    result.update({"epoch":int(epoch_1based),"lr":float(lr),"train_seconds":elapsed,
                   "train_examples_per_sec":n/elapsed,"train_peak_rss_bytes":int(rss.peak),
                   "train_peak_gpu_allocated_bytes":gpu["peak_allocated_bytes"],
                   "train_peak_gpu_reserved_bytes":gpu["peak_reserved_bytes"]})
    if grad_values:
        gv=torch.stack(grad_values).float(); result["grad_norm_max"]=float(gv.max().cpu()); result["grad_norm_median"]=float(gv.median().cpu())
        result["grad_norm_values"]=[float(v) for v in gv.cpu().tolist()]
    return result


@torch.no_grad()
def evaluate_selection_metric(model,bundle,resolution,k_train,method,model_seed):
    model.eval(); loader=make_loader(bundle.val_images,bundle.val_labels,resolution,FIXED["batch_size"],False,
                                     stable_seed("val_loader",bundle.name,resolution,model_seed))
    gen=new_torch_generator(stable_seed("val_mc_common",bundle.name,bundle.split_name,resolution,model_seed,k_train))
    total=torch.tensor(0.0,device=DEVICE); recon=torch.tensor(0.0,device=DEVICE); kl=torch.tensor(0.0,device=DEVICE); n=0; mus=[]
    base_gpu=_gpu_peak_reset(); cuda_sync(); t0=time.perf_counter()
    with PeakRSSMonitor(FIXED["ram_poll_interval_sec"]) as rss:
        for x,q,_ in loader:
            x=x.to(DEVICE,non_blocking=True); q=q.to(DEVICE,non_blocking=True); bs=x.shape[0]; n+=bs
            mu,log_std=model.encode(x); mus.append(mu.detach().cpu())
            if method=="vae":
                z=model.reparameterize(mu,log_std,1,generator=gen); log_px=observation_log_prob(model,model.decode(z),q).squeeze(0)
                per=-(log_px-analytic_kl_standard_normal(mu,log_std)); recon += (-log_px).sum(); kl += analytic_kl_standard_normal(mu,log_std).sum()
            else:
                parts=compute_log_weight_parts(model,x,q,k_train,generator=gen)
                per=-(torch.logsumexp(parts["log_w"],0)-math.log(k_train)); recon += (-parts["log_px"].mean(0)).sum(); kl += analytic_kl_standard_normal(mu,log_std).sum()
            total += per.sum()
    cuda_sync(); elapsed=time.perf_counter()-t0; gpu=_gpu_peak_read(base_gpu)
    mu_all=torch.cat(mus,0); active=int((mu_all.var(0,unbiased=False)>FIXED["active_unit_threshold"]).sum())
    return {"val_selection_bpd":float((total/n/(LN2*PIXELS)).cpu()),"val_recon_bpd":float((recon/n/(LN2*PIXELS)).cpu()),
            "val_analytic_kl_bpd":float((kl/n/(LN2*PIXELS)).cpu()),"active_units":active,"val_seconds":elapsed,
            "val_examples_per_sec":n/elapsed,"val_peak_rss_bytes":int(rss.peak),"val_peak_gpu_allocated_bytes":gpu["peak_allocated_bytes"],
            "val_peak_gpu_reserved_bytes":gpu["peak_reserved_bytes"]}


@torch.no_grad()
def evaluate_importance_bound(model,images,labels,resolution,k_eval,seed,batch_size=None):
    """Common-K stochastic lower-bound evaluation. Returned 'negative_bound' is not called exact NLL."""
    model.eval(); batch_size=batch_size or FIXED["batch_size"]
    loader=make_loader(images,labels,resolution,batch_size,False,stable_seed("eval_loader",seed))
    gen=new_torch_generator(seed); neg_sum=torch.tensor(0.0,device=DEVICE); recon_sum=torch.tensor(0.0,device=DEVICE); kl_sum=torch.tensor(0.0,device=DEVICE); n=0; mus=[]
    chunk=int(FIXED["eval_k_chunk_size"]); base_gpu=_gpu_peak_reset(); cuda_sync(); t0=time.perf_counter()
    with PeakRSSMonitor(FIXED["ram_poll_interval_sec"]) as rss:
        for x,q,_ in loader:
            x=x.to(DEVICE,non_blocking=True); q=q.to(DEVICE,non_blocking=True); bs=x.shape[0]; n+=bs
            mu,log_std=model.encode(x); mus.append(mu.cpu()); logsum=None; recon_batch=torch.zeros(bs,device=DEVICE); done=0
            while done<int(k_eval):
                kc=min(chunk,int(k_eval)-done); z=model.reparameterize(mu,log_std,kc,generator=gen); dec=model.decode(z)
                log_px=observation_log_prob(model,dec,q); log_w=log_px+standard_normal_log_prob(z)-diag_normal_log_prob(z,mu.unsqueeze(0),log_std.unsqueeze(0))
                lse=torch.logsumexp(log_w,0); logsum=lse if logsum is None else torch.logaddexp(logsum,lse)
                recon_batch += (-log_px).sum(0); done+=kc
            neg_sum += (-(logsum-math.log(k_eval))).sum(); recon_sum += (recon_batch/k_eval).sum(); kl_sum += analytic_kl_standard_normal(mu,log_std).sum()
    cuda_sync(); elapsed=time.perf_counter()-t0; gpu=_gpu_peak_read(base_gpu); mu_all=torch.cat(mus,0)
    active=int((mu_all.var(0,unbiased=False)>FIXED["active_unit_threshold"]).sum())
    return {"negative_iwae_bound_nats_per_image":float((neg_sum/n).cpu()),"bpd_from_Lk":float((neg_sum/n/(LN2*PIXELS)).cpu()),
            "reconstruction_bpd_diagnostic":float((recon_sum/n/(LN2*PIXELS)).cpu()),"analytic_kl_bpd_diagnostic":float((kl_sum/n/(LN2*PIXELS)).cpu()),
            "active_units":active,"seconds":elapsed,"examples_per_sec":n/elapsed,"peak_rss_bytes":int(rss.peak),
            "peak_gpu_allocated_bytes":gpu["peak_allocated_bytes"],"peak_gpu_reserved_bytes":gpu["peak_reserved_bytes"]}


@torch.no_grad()
def benchmark_full_k1_density_path(model,bundle,resolution,model_seed):
    """Latency of the full K=1 log-weight path with epsilon pre-generated so RNG cost is excluded."""
    model.eval(); bs=min(FIXED["benchmark_batch_size"],len(bundle.val_images)); ds=QuantizedTensorDataset(bundle.val_images[:bs],bundle.val_labels[:bs],resolution)
    batch=[ds[i] for i in range(bs)]; x=torch.stack([a[0] for a in batch]).to(DEVICE); q=torch.stack([a[1] for a in batch]).to(DEVICE)
    eps=torch.randn((1,bs,FIXED["latent_dim"]),device=DEVICE,generator=new_torch_generator(stable_seed("bench_eps",bundle.name,resolution,model_seed)))
    def call():
        mu,ls=model.encode(x); z=mu.unsqueeze(0)+torch.exp(ls).unsqueeze(0)*eps; lp=observation_log_prob(model,model.decode(z),q)
        return lp+standard_normal_log_prob(z)-diag_normal_log_prob(z,mu.unsqueeze(0),ls.unsqueeze(0))
    for _ in range(FIXED["benchmark_warmup"]): call()
    cuda_sync(); times=[]
    for _ in range(FIXED["benchmark_repetitions"]):
        cuda_sync(); t=time.perf_counter(); y=call(); cuda_sync(); times.append(time.perf_counter()-t)
        if not torch.isfinite(y).all(): raise FloatingPointError("Nonfinite benchmark output")
    a=np.asarray(times)
    return {"k1_full_density_batch_size":bs,"k1_full_density_latency_ms_mean":float(a.mean()*1000),
            "k1_full_density_latency_ms_median":float(np.median(a)*1000),"k1_full_density_latency_ms_p95":float(np.quantile(a,.95)*1000),
            "k1_full_density_examples_per_sec":float(bs/a.mean())}

## Implementation and protocol fingerprints

The resume system hashes executable semantics rather than trusting a manually bumped version string. The training fingerprint intentionally excludes experimental levels but includes fixed protocol, source-plan hash, and the core implementation. Dataset and software fingerprints are checked separately.

In [ ]:
def _code_semantics(code_obj):
    out={"co_code":code_obj.co_code.hex(),"co_names":list(code_obj.co_names),"co_varnames":list(code_obj.co_varnames),"consts":[]}
    for c in code_obj.co_consts:
        if isinstance(c,types.CodeType): out["consts"].append(_code_semantics(c))
        elif isinstance(c,(str,int,float,bool,type(None))): out["consts"].append(c)
        else: out["consts"].append(repr(c))
    return out

def semantic_object_hash(objects):
    payload=[]
    for obj in objects:
        if inspect.isclass(obj):
            methods={k:_code_semantics(v.__code__) for k,v in obj.__dict__.items() if callable(v) and hasattr(v,"__code__")}
            payload.append({"class":obj.__name__,"methods":methods})
        elif hasattr(obj,"__code__"): payload.append({"function":obj.__name__,"code":_code_semantics(obj.__code__)})
        else: payload.append(repr(obj))
    return sha256_json(payload)

TRAINING_IMPLEMENTATION_FINGERPRINT=semantic_object_hash([
    QuantizedTensorDataset,LatentDensityMLP,discretized_logistic_log_mass,observation_log_prob,compute_log_weight_parts,
    vae_training_parts,train_step,train_one_epoch,evaluate_selection_metric,evaluate_importance_bound,benchmark_full_k1_density_path,
    make_loader,make_optimizer,apply_lr_schedule,stable_seed,set_global_seed,seed_worker
])
TRAINING_PROTOCOL_FINGERPRINT=sha256_json({"source_spec_sha256":SOURCE_SPEC_SHA256,"fixed":FIXED,"pilot_acceptance":PILOT,"implementation":TRAINING_IMPLEMENTATION_FINGERPRINT})
STUDY_DESIGN_FINGERPRINT=sha256_json({"source_spec_sha256":SOURCE_SPEC_SHA256,"experimental":EXPERIMENTAL,"fixed":FIXED,"pilot":PILOT,"emnist":EMNIST_CONFIG})
print("Training implementation fingerprint:",TRAINING_IMPLEMENTATION_FINGERPRINT)
print("Training protocol fingerprint:",TRAINING_PROTOCOL_FINGERPRINT)
print("Study design fingerprint:",STUDY_DESIGN_FINGERPRINT)

Training implementation fingerprint: 9b1ff29e9915b71fda94f14077f7e16a778635013d9a6eda517447560b4e6b9b
Training protocol fingerprint: 2fdc0b19ac16e1161c60759a1c44ced0566886ad5efe711cb2ff678922a16259
Study design fingerprint: e20cd15d5b682615121ff80c4bc1202222dc5539adb2e703977934eb0956f3a8


In [ ]:
@dataclass(frozen=True)
class RunSpec:
    phase: str
    dataset: str
    split_name: str
    resolution: int
    k_train: int
    method: str
    seed: int
    def experiment_id(self): return f"{self.phase}_{self.dataset}-{self.split_name}_r{self.resolution}_{self.method}_k{self.k_train}_z20_seed{self.seed}"


def grid_specs(phase,dataset,split_name,resolutions=None,k_values=None,estimators=None,seeds=None):
    resolutions=list(EXPERIMENTAL["observation_resolutions"] if resolutions is None else resolutions)
    k_values=list(EXPERIMENTAL["k_train_values"] if k_values is None else k_values)
    estimators=list(EXPERIMENTAL["estimators_k_gt_1"] if estimators is None else estimators)
    seeds=list(EXPERIMENTAL["seeds"] if seeds is None else seeds); out=[]
    for r in resolutions:
        for k in k_values:
            methods=["vae"] if int(k)==1 else estimators
            for m in methods:
                for s in seeds: out.append(RunSpec(phase,dataset,split_name,int(r),int(k),m,int(s)))
    return out

PILOT_SPECS=grid_specs("pilot","mnist","official",PILOT["resolutions"],PILOT["k_values"],PILOT["estimators_k_gt_1"],[PILOT["seed"]]) if PILOT["enabled"] else []
MNIST_SPECS=grid_specs("main","mnist","official")
EMNIST_SPECS=grid_specs("generalization","emnist",EMNIST_CONFIG["split"])
assert all(s.method=="vae" for s in MNIST_SPECS if s.k_train==1)
if set(EXPERIMENTAL["observation_resolutions"])=={2,16,256} and set(EXPERIMENTAL["k_train_values"])=={1,5,10,20,50} and len(EXPERIMENTAL["seeds"])==3 and set(EXPERIMENTAL["estimators_k_gt_1"])=={"iwae","dreg"}:
    assert len(MNIST_SPECS)==81 and len(EMNIST_SPECS)==81
print("Expected pilot/main/EMNIST runs:",len(PILOT_SPECS),len(MNIST_SPECS),len(EMNIST_SPECS))


def selected_for_shard(spec): return stable_seed("shard",spec.experiment_id()) % RUN_CONTROL["num_shards"] == RUN_CONTROL["shard_index"]
def execution_subset(specs):
    selected=[s for s in specs if selected_for_shard(s)]
    if RUN_CONTROL["debug_max_runs"] is not None: selected=selected[:int(RUN_CONTROL["debug_max_runs"])]
    return selected

GRID_MANIFEST=pd.DataFrame([{**asdict(s),"experiment_id":s.experiment_id(),"selected_current_shard":selected_for_shard(s)} for s in PILOT_SPECS+MNIST_SPECS+EMNIST_SPECS])
GRID_MANIFEST.to_csv(EXPORT_ROOT/"expected_grid_manifest.csv",index=False); display(GRID_MANIFEST.groupby(["phase","selected_current_shard"]).size().rename("runs"))
CONFIG_INTEGRITY = {
    "grid_R_matches_master": {s.resolution for s in MNIST_SPECS} == set(EXPERIMENTAL["observation_resolutions"]),
    "grid_K_matches_master": {s.k_train for s in MNIST_SPECS} == set(EXPERIMENTAL["k_train_values"]),
    "grid_seeds_match_master": {s.seed for s in MNIST_SPECS} == set(EXPERIMENTAL["seeds"]),
    "K1_only_VAE": all(s.method=="vae" for s in MNIST_SPECS if s.k_train==1),
    "K_gt_1_estimators_match_master": {s.method for s in MNIST_SPECS if s.k_train>1} == set(EXPERIMENTAL["estimators_k_gt_1"]),
    "latent_not_in_RunSpec_and_fixed_20": "latent_dim" not in RunSpec.__dataclass_fields__ and FIXED["latent_dim"]==20,
}
assert all(CONFIG_INTEGRITY.values()), CONFIG_INTEGRITY
display(pd.DataFrame([CONFIG_INTEGRITY]).T.rename(columns={0:"pass"}))


Expected pilot/main/EMNIST runs: 10 81 81


phase           selected_current_shard
generalization  True                      81
main            True                      81
pilot           True                      10
Name: runs, dtype: int64

,pass
grid_R_matches_master,True
grid_K_matches_master,True
grid_seeds_match_master,True
K1_only_VAE,True
K_gt_1_estimators_match_master,True
latent_not_in_RunSpec_and_fixed_20,True


In [ ]:
def atomic_torch_save(obj,path):
    path=Path(path); path.parent.mkdir(parents=True,exist_ok=True); tmp=path.with_suffix(path.suffix+".tmp"); torch.save(obj,tmp); os.replace(tmp,path)
def atomic_json(obj,path):
    path=Path(path); path.parent.mkdir(parents=True,exist_ok=True); tmp=path.with_suffix(path.suffix+".tmp")
    with open(tmp,"w") as f: json.dump(obj,f,indent=2,default=str,allow_nan=False)
    os.replace(tmp,path)

def _normalized_roots():
    roots=[OUTPUT_ROOT]
    for raw in RUN_CONTROL["resume_roots"]:
        p=Path(raw)
        if not (p/"experiments").exists() and (p/RUN_CONTROL["run_tag"]/"experiments").exists(): p=p/RUN_CONTROL["run_tag"]
        roots.append(p)
    return roots

def _candidate_attempt_dirs(spec):
    out=[]
    for root in _normalized_roots():
        base=root/"experiments"/spec.experiment_id()
        if base.exists(): out.extend(sorted([p for p in base.iterdir() if p.is_dir()],reverse=True))
    return out

def expected_run_metadata(spec,bundle):
    return {"experiment_id":spec.experiment_id(),"spec":asdict(spec),"latent_dim":20,"source_spec_sha256":SOURCE_SPEC_SHA256,
            "training_implementation_fingerprint":TRAINING_IMPLEMENTATION_FINGERPRINT,"training_protocol_fingerprint":TRAINING_PROTOCOL_FINGERPRINT,"study_design_fingerprint":STUDY_DESIGN_FINGERPRINT,
            "dataset_fingerprint":bundle.dataset_fingerprint,"software_fingerprint":SOFTWARE_FINGERPRINT,"benchmark_environment_fingerprint":BENCHMARK_ENVIRONMENT_FINGERPRINT}

def compatible_metadata(saved,spec,bundle,require_software=True):
    exp=expected_run_metadata(spec,bundle)
    keys=["experiment_id","source_spec_sha256","training_implementation_fingerprint","training_protocol_fingerprint","dataset_fingerprint"]
    if require_software: keys.append("software_fingerprint")
    return all(saved.get(k)==exp.get(k) for k in keys)

def find_completed(spec,bundle):
    if not RUN_CONTROL["resume_completed"]: return None
    for d in _candidate_attempt_dirs(spec):
        rp=d/"result.json"; cp=d/"config.json"
        if rp.exists() and cp.exists():
            try:
                saved=json.load(open(cp)); result=json.load(open(rp))
                if result.get("status")=="complete" and compatible_metadata(saved,spec,bundle,RUN_CONTROL["require_uniform_software_for_scientific_analysis"]):
                    result=dict(result); result["artifact_dir"]=str(d); return d,result
            except Exception: pass
    return None

def find_incomplete(spec,bundle):
    if not RUN_CONTROL["resume_incomplete"]: return None
    for d in _candidate_attempt_dirs(spec):
        cp=d/"config.json"; lp=d/"last_checkpoint.pt"
        if cp.exists() and lp.exists() and not (d/"result.json").exists():
            try:
                saved=json.load(open(cp))
                if compatible_metadata(saved,spec,bundle,True) and saved.get("benchmark_environment_fingerprint")==BENCHMARK_ENVIRONMENT_FINGERPRINT: return d
            except Exception: pass
    return None

def new_attempt_dir(spec):
    base=OUTPUT_ROOT/"experiments"/spec.experiment_id(); base.mkdir(parents=True,exist_ok=True)
    attempt=datetime.now(timezone.utc).strftime("attempt_%Y%m%dT%H%M%S_%fZ")
    d=base/attempt; d.mkdir(parents=False,exist_ok=False); return d

In [ ]:
def run_experiment(spec: RunSpec, collect_pilot_gradients=False):
    if FIXED["latent_dim"]!=20: raise AssertionError("latent dimension must remain 20")
    bundle=DATASETS[spec.dataset]
    completed=find_completed(spec,bundle)
    if completed is not None:
        d,result=completed; print("REUSE",spec.experiment_id(),"from",d); return result

    prior=find_incomplete(spec,bundle); run_dir=new_attempt_dir(spec); meta=expected_run_metadata(spec,bundle)
    meta.update({"fixed":FIXED,"experimental":EXPERIMENTAL,"emnist_config":EMNIST_CONFIG,"environment":ENVIRONMENT,"run_dir":str(run_dir)})
    atomic_json(meta,run_dir/"config.json")
    model_init_seed=stable_seed("model_init",spec.dataset,spec.split_name,spec.resolution,spec.seed)
    set_global_seed(model_init_seed); model=LatentDensityMLP(spec.resolution).to(DEVICE); optimizer,opt_kwargs=make_optimizer(model)
    history=[]; start_epoch=1; best_bpd=float("inf"); best_epoch=None; best_cumulative_fit_seconds=None; pilot_grad_values=[]
    if prior is not None:
        ck=torch.load(prior/"last_checkpoint.pt",map_location=DEVICE,weights_only=False); model.load_state_dict(ck["model"]); optimizer.load_state_dict(ck["optimizer"])
        history=ck["history"]; start_epoch=int(ck["epoch"])+1; best_bpd=float(ck["best_bpd"]); best_epoch=ck["best_epoch"]; best_cumulative_fit_seconds=ck.get("best_cumulative_fit_seconds")
        if (prior/"best_checkpoint.pt").exists(): shutil.copy2(prior/"best_checkpoint.pt",run_dir/"best_checkpoint.pt")
        pg=prior/"pilot_grad_norms.json"
        if pg.exists(): pilot_grad_values=json.load(open(pg))
        print("RESUME",spec.experiment_id(),"after epoch",start_epoch-1,"from",prior)
    total_train=float(sum(float(x.get("train_seconds",0.0)) for x in history)); total_val=float(sum(float(x.get("val_seconds",0.0)) for x in history))
    train_gpu=max([int(x.get("train_peak_gpu_allocated_bytes",0)) for x in history] or [0]); val_gpu=max([int(x.get("val_peak_gpu_allocated_bytes",0)) for x in history] or [0])
    train_gpu_reserved=max([int(x.get("train_peak_gpu_reserved_bytes",0)) for x in history] or [0]); val_gpu_reserved=max([int(x.get("val_peak_gpu_reserved_bytes",0)) for x in history] or [0])
    train_rss=max([int(x.get("train_peak_rss_bytes",0)) for x in history] or [0]); val_rss=max([int(x.get("val_peak_rss_bytes",0)) for x in history] or [0])
    try:
        for epoch in range(start_epoch,FIXED["epochs"]+1):
            tr=train_one_epoch(model,optimizer,bundle,spec.resolution,spec.k_train,spec.method,spec.seed,epoch,collect_pilot_gradients)
            va=evaluate_selection_metric(model,bundle,spec.resolution,spec.k_train,spec.method,spec.seed)
            if not all(np.isfinite(v) for k,v in {**tr,**va}.items() if isinstance(v,(int,float))): raise FloatingPointError("Nonfinite epoch metric")
            total_train += tr["train_seconds"]; total_val += va["val_seconds"]
            train_gpu=max(train_gpu,tr["train_peak_gpu_allocated_bytes"]); val_gpu=max(val_gpu,va["val_peak_gpu_allocated_bytes"])
            train_gpu_reserved=max(train_gpu_reserved,tr["train_peak_gpu_reserved_bytes"]); val_gpu_reserved=max(val_gpu_reserved,va["val_peak_gpu_reserved_bytes"])
            train_rss=max(train_rss,tr["train_peak_rss_bytes"]); val_rss=max(val_rss,va["val_peak_rss_bytes"])
            row={k:v for k,v in tr.items() if k!="grad_norm_values"}; row.update(va); row["cumulative_fit_seconds"]=total_train+total_val; history.append(row)
            if collect_pilot_gradients and "grad_norm_values" in tr:
                pilot_grad_values.extend(tr["grad_norm_values"]); atomic_json(pilot_grad_values,run_dir/"pilot_grad_norms.json")
            if va["val_selection_bpd"] < best_bpd:
                best_bpd=va["val_selection_bpd"]; best_epoch=epoch; best_cumulative_fit_seconds=total_train+total_val
                atomic_torch_save({"model":model.state_dict(),"epoch":epoch,"val_selection_bpd":best_bpd},run_dir/"best_checkpoint.pt")
            pd.DataFrame(history).to_csv(run_dir/"history.csv",index=False)
            atomic_torch_save({"model":model.state_dict(),"optimizer":optimizer.state_dict(),"epoch":epoch,"history":history,"best_bpd":best_bpd,
                               "best_epoch":best_epoch,"best_cumulative_fit_seconds":best_cumulative_fit_seconds},run_dir/"last_checkpoint.pt")
            print(f"{spec.experiment_id()} epoch {epoch:03d} train={tr['objective_bpd']:.4f} val={va['val_selection_bpd']:.4f}")
        best=torch.load(run_dir/"best_checkpoint.pt",map_location=DEVICE,weights_only=False); model.load_state_dict(best["model"])
        common_val=evaluate_importance_bound(model,bundle.val_images,bundle.val_labels,spec.resolution,FIXED["k_eval"],
                    stable_seed("common_eval",spec.dataset,"val",spec.resolution,spec.seed,FIXED["k_eval"]))
        common_test=evaluate_importance_bound(model,bundle.test_images,bundle.test_labels,spec.resolution,FIXED["k_eval"],
                    stable_seed("common_eval",spec.dataset,"test",spec.resolution,spec.seed,FIXED["k_eval"]))
        bench=benchmark_full_k1_density_path(model,bundle,spec.resolution,spec.seed)
        total_params,trainable_params,param_bytes=parameter_metrics(model); mac=linear_macs_per_example(spec.resolution,spec.k_train)
        ckpt_bytes=(run_dir/"best_checkpoint.pt").stat().st_size
        mac["forward_linear_macs_per_training_epoch"] = int(mac["forward_linear_macs_per_example"] * len(bundle.train_images))
        mac["forward_linear_macs_full_epoch_budget"] = int(mac["forward_linear_macs_per_training_epoch"] * FIXED["epochs"])
        result={**meta,"status":"complete","optimizer_actual_kwargs":opt_kwargs,"best_epoch":best_epoch,"best_val_selection_bpd":best_bpd,
                "best_cumulative_fit_seconds":best_cumulative_fit_seconds,"total_train_seconds":total_train,"total_validation_selection_seconds":total_val,
                "total_fit_selection_seconds":total_train+total_val,"train_peak_gpu_allocated_bytes":train_gpu,"validation_peak_gpu_allocated_bytes":val_gpu,
                "train_peak_gpu_reserved_bytes":train_gpu_reserved,"validation_peak_gpu_reserved_bytes":val_gpu_reserved,
                "train_peak_rss_bytes":train_rss,"validation_peak_rss_bytes":val_rss,"parameters":total_params,"trainable_parameters":trainable_params,
                "parameter_bytes":param_bytes,"best_checkpoint_bytes":ckpt_bytes,**mac,**bench}
        for prefix,obj in [("val_k_eval",common_val),("test_k_eval",common_test)]:
            for k,v in obj.items(): result[f"{prefix}_{k}"]=v
        if collect_pilot_gradients and pilot_grad_values:
            gv=np.asarray(pilot_grad_values,float); result["pilot_grad_norm_max"]=float(np.max(gv)); result["pilot_grad_norm_median"]=float(np.median(gv));
            result["pilot_grad_spike_ratio"]=float(np.max(gv)/max(np.median(gv),1e-12))
        result["artifact_dir"]=str(run_dir)
        atomic_json(result,run_dir/"result.json"); return result
    except BaseException as e:
        atomic_json({"status":"failed","experiment_id":spec.experiment_id(),"error":repr(e),"traceback":traceback.format_exc()},run_dir/"failure.json")
        raise


def execute_specs(specs,collect_pilot_gradients=False):
    out=[]
    for spec in execution_subset(specs):
        try: out.append(run_experiment(spec,collect_pilot_gradients))
        except Exception as e:
            print("FAILED",spec.experiment_id(),repr(e))
            if not RUN_CONTROL["continue_after_run_failure"]: raise
    return out# Extend the implementation fingerprint to include experiment orchestration and resume semantics.
CORE_TRAINING_IMPLEMENTATION_FINGERPRINT=TRAINING_IMPLEMENTATION_FINGERPRINT
RUNNER_IMPLEMENTATION_FINGERPRINT=semantic_object_hash([
    expected_run_metadata,compatible_metadata,find_completed,find_incomplete,new_attempt_dir,
    run_experiment,execute_specs,atomic_torch_save,atomic_json
])
TRAINING_IMPLEMENTATION_FINGERPRINT=sha256_json({
    "core":CORE_TRAINING_IMPLEMENTATION_FINGERPRINT,
    "runner":RUNNER_IMPLEMENTATION_FINGERPRINT,
})
TRAINING_PROTOCOL_FINGERPRINT=sha256_json({
    "source_spec_sha256":SOURCE_SPEC_SHA256,"fixed":FIXED,"pilot_acceptance":PILOT,"implementation":TRAINING_IMPLEMENTATION_FINGERPRINT
})
STUDY_DESIGN_FINGERPRINT=sha256_json({"source_spec_sha256":SOURCE_SPEC_SHA256,"experimental":EXPERIMENTAL,"fixed":FIXED,"pilot":PILOT,"emnist":EMNIST_CONFIG})
print("Final runner-aware implementation fingerprint:",TRAINING_IMPLEMENTATION_FINGERPRINT)
print("Final training protocol fingerprint:",TRAINING_PROTOCOL_FINGERPRINT)
print("Final study design fingerprint:",STUDY_DESIGN_FINGERPRINT)

Final runner-aware implementation fingerprint: 41c0caf494b68e15488d5d5309d3d3bd5b955ca97574b0e547d52bf08321bfb5
Final training protocol fingerprint: 986f50280ca42fbee0b761f4603fe1f4ab41dd4df1fc31226ffaae17e3540d2b
Final study design fingerprint: e20cd15d5b682615121ff80c4bc1202222dc5539adb2e703977934eb0956f3a8


## Pilot → freeze gate

The pilot gate is **fail-closed**. It never changes a hyperparameter. A failing pilot exports diagnostics and stops before the main grid; the investigator must revise only the MASTER CONFIGURATION, document the new fixed protocol, and rerun the pilot. This is the only scientifically honest way to preserve the plan's “pilot, then freeze” requirement without outcome-driven hidden tuning.

In [ ]:
def collect_completed(specs):
    rows=[]; missing=[]
    for s in specs:
        b=DATASETS.get(s.dataset)
        if b is None: missing.append(s.experiment_id()); continue
        found=find_completed(s,b)
        if found is None: missing.append(s.experiment_id())
        else: rows.append(found[1])
    return rows,missing


def evaluate_pilot_gate(rows,specs=None):
    specs=PILOT_SPECS if specs is None else list(specs)
    byid={r["experiment_id"]:r for r in rows}; decisions=[]
    for s in specs:
        r=byid.get(s.experiment_id()); reasons=[]
        if r is None:
            reasons.append("missing completed pilot run")
        else:
            hp=Path(r.get("artifact_dir",r["run_dir"]))/ "history.csv"
            h=pd.read_csv(hp) if hp.exists() else pd.DataFrame()

            # The pilot gate is a safety/stability check, not a convergence-speed gate.
            # A resolution is allowed to keep improving late in training because that
            # may be a legitimate experimental effect rather than a failure condition.
            if len(h)!=FIXED["epochs"]:
                reasons.append("incomplete epoch history")

            if len(h)>=2*PILOT["late_window_epochs"]:
                w=PILOT["late_window_epochs"]
                prev=h["val_selection_bpd"].iloc[-2*w:-w].mean()
                last=h["val_selection_bpd"].iloc[-w:].mean()
                regression=float(last-prev)

                # Fail only on substantial late validation degradation.
                if regression>PILOT["max_late_regression_bpd"]:
                    reasons.append(f"late validation regression {regression:.6g} exceeds gate")
            else:
                reasons.append("insufficient epochs for late-window gate")

            spike=r.get("pilot_grad_spike_ratio")
            if spike is None or not np.isfinite(spike):
                reasons.append("missing/nonfinite gradient-spike diagnostic")
            elif spike>PILOT["max_gradient_spike_ratio"]:
                reasons.append(f"gradient spike ratio {spike:.3g} exceeds gate")

            # Explicit finite-value safety check for the recorded validation metric.
            if not h.empty and not np.isfinite(h["val_selection_bpd"].to_numpy(dtype=float)).all():
                reasons.append("nonfinite validation bpd detected")

        decisions.append({
            "experiment_id":s.experiment_id(),
            "pass":len(reasons)==0,
            "reasons":"; ".join(reasons)
        })

    df=pd.DataFrame(decisions)
    return bool(len(df)==len(specs) and df["pass"].all()),df

PILOT_EXECUTED=[]
if PILOT["enabled"]:
    if RUN_CONTROL["shard_index"]==PILOT["execute_on_shard_index"]:
        binary_specs=[s for s in PILOT_SPECS if s.resolution==2]
        later_specs=[s for s in PILOT_SPECS if s.resolution!=2]
        PILOT_EXECUTED.extend(execute_specs(binary_specs,collect_pilot_gradients=True))
        binary_rows,binary_missing=collect_completed(binary_specs); binary_pass,binary_gate=evaluate_pilot_gate(binary_rows,binary_specs)
        binary_gate.to_csv(EXPORT_ROOT/"pilot_binary_gate.csv",index=False)
        if not binary_pass:
            raise RuntimeError("Binary R=2 pilot gate failed/incomplete; R=256 pilot intentionally withheld.")
        PILOT_EXECUTED.extend(execute_specs(later_specs,collect_pilot_gradients=True))
    pilot_rows,pilot_missing=collect_completed(PILOT_SPECS); PILOT_PASSED,PILOT_GATE_TABLE=evaluate_pilot_gate(pilot_rows,PILOT_SPECS)
    PILOT_GATE_TABLE.to_csv(EXPORT_ROOT/"pilot_gate.csv",index=False); atomic_json({"passed":PILOT_PASSED,"missing":pilot_missing,"criteria":PILOT},EXPORT_ROOT/"pilot_gate.json")
    # Pilot compute-budget refinement: interpolate measured seconds/epoch across measured K values.
    timing=[]
    for r in pilot_rows:
        hp=Path(r.get("artifact_dir",r["run_dir"]))/"history.csv"
        if hp.exists():
            h=pd.read_csv(hp); timing.append({"k_train":r["spec"]["k_train"],"seconds_per_epoch":float(h["train_seconds"].mean())})
    if timing:
        pt=pd.DataFrame(timing).groupby("k_train",as_index=False).seconds_per_epoch.mean().sort_values("k_train")
        ks=pt.k_train.to_numpy(float); ts=pt.seconds_per_epoch.to_numpy(float)
        est=sum(float(np.interp(s.k_train,ks,ts))*FIXED["epochs"] for s in MNIST_SPECS)
        PILOT_COMPUTE_ESTIMATE={"measured_seconds_per_epoch_by_k":pt.to_dict("records"),"estimated_main_grid_train_seconds":est,"estimated_main_grid_train_gpu_hours":est/3600.0,"note":"Interpolation-based planning estimate only; final efficiency uses measured run times."}
        atomic_json(PILOT_COMPUTE_ESTIMATE,EXPORT_ROOT/"pilot_compute_estimate.json"); display(pd.DataFrame([PILOT_COMPUTE_ESTIMATE]))
    display(PILOT_GATE_TABLE)
    if not PILOT_PASSED:
        raise RuntimeError("PILOT GATE FAILED or is incomplete. Main grid intentionally withheld. Inspect pilot_gate.csv, revise only MASTER CONFIG if scientifically justified, and rerun.")
else:
    PILOT_PASSED=True
print("Pilot gate: PASS")

REUSE pilot_mnist-official_r2_vae_k1_z20_seed1 from /home/imohammedali/projects/pgm/latent_density_results/research_grade_z20_v2/experiments/pilot_mnist-official_r2_vae_k1_z20_seed1/attempt_20260905T214740_098639Z
REUSE pilot_mnist-official_r2_iwae_k10_z20_seed1 from /home/imohammedali/projects/pgm/latent_density_results/research_grade_z20_v2/experiments/pilot_mnist-official_r2_iwae_k10_z20_seed1/attempt_20260906T105058_015939Z
REUSE pilot_mnist-official_r2_dreg_k10_z20_seed1 from /home/imohammedali/projects/pgm/latent_density_results/research_grade_z20_v2/experiments/pilot_mnist-official_r2_dreg_k10_z20_seed1/attempt_20260905T215607_317497Z
REUSE pilot_mnist-official_r2_iwae_k50_z20_seed1 from /home/imohammedali/projects/pgm/latent_density_results/research_grade_z20_v2/experiments/pilot_mnist-official_r2_iwae_k50_z20_seed1/attempt_20260905T220055_930822Z
REUSE pilot_mnist-official_r2_dreg_k50_z20_seed1 from /home/imohammedali/projects/pgm/latent_density_results/research_grade_z20_v2/e

,measured_seconds_per_epoch_by_k,estimated_main_grid_train_seconds,estimated_main_grid_train_gpu_hours,note
0,"[{'k_train': 1, 'seconds_per_epoch': 2.2927216...",21634.376401,6.009549,Interpolation-based planning estimate only; fi...


,experiment_id,pass,reasons
0,pilot_mnist-official_r2_vae_k1_z20_seed1,True,
1,pilot_mnist-official_r2_iwae_k10_z20_seed1,True,
2,pilot_mnist-official_r2_dreg_k10_z20_seed1,True,
3,pilot_mnist-official_r2_iwae_k50_z20_seed1,True,
4,pilot_mnist-official_r2_dreg_k50_z20_seed1,True,
5,pilot_mnist-official_r256_vae_k1_z20_seed1,True,
6,pilot_mnist-official_r256_iwae_k10_z20_seed1,True,
7,pilot_mnist-official_r256_dreg_k10_z20_seed1,True,
8,pilot_mnist-official_r256_iwae_k50_z20_seed1,True,
9,pilot_mnist-official_r256_dreg_k50_z20_seed1,True,


Pilot gate: PASS


# 10. Evaluation

Checkpoint selection uses validation only. After the selected checkpoint is restored, **every run** receives a common-`K_eval=1000` validation evaluation and only then its final test evaluation. The common-K validation result—not test performance—is later used to select a single “best configuration.”

In [ ]:
MAIN_EXECUTED=[]
if PHASES["run_mnist_main_grid"]:
    MAIN_EXECUTED=execute_specs(MNIST_SPECS,collect_pilot_gradients=False)

MAIN_ROWS,MAIN_MISSING=collect_completed(MNIST_SPECS)
MAIN_COMPLETE=(len(MAIN_MISSING)==0)
pd.DataFrame({"missing_experiment_id":MAIN_MISSING}).to_csv(EXPORT_ROOT/"mnist_missing_runs.csv",index=False)
print(f"MNIST main complete: {MAIN_COMPLETE}; completed={len(MAIN_ROWS)}/{len(MNIST_SPECS)}")

REUSE main_mnist-official_r2_vae_k1_z20_seed1 from /home/imohammedali/projects/pgm/latent_density_results/research_grade_z20_v2/experiments/main_mnist-official_r2_vae_k1_z20_seed1/attempt_20260906T043840_907364Z
REUSE main_mnist-official_r2_vae_k1_z20_seed2 from /home/imohammedali/projects/pgm/latent_density_results/research_grade_z20_v2/experiments/main_mnist-official_r2_vae_k1_z20_seed2/attempt_20260906T174150_183901Z
REUSE main_mnist-official_r2_vae_k1_z20_seed3 from /home/imohammedali/projects/pgm/latent_density_results/research_grade_z20_v2/experiments/main_mnist-official_r2_vae_k1_z20_seed3/attempt_20260906T044615_114701Z
REUSE main_mnist-official_r2_iwae_k5_z20_seed1 from /home/imohammedali/projects/pgm/latent_density_results/research_grade_z20_v2/experiments/main_mnist-official_r2_iwae_k5_z20_seed1/attempt_20260906T045007_675338Z
REUSE main_mnist-official_r2_iwae_k5_z20_seed2 from /home/imohammedali/projects/pgm/latent_density_results/research_grade_z20_v2/experiments/main_mnis

# 11. Computational Benchmarking

Per-run records include train-only and validation-only wall time, throughput, parameter counts, parameter/checkpoint size, analytical dense-layer MACs/FLOPs, separately measured training/validation/test GPU and RSS peaks, common-K evaluation throughput, and warmed repeated full K=1 density latency. Comparative efficiency conclusions are suppressed if benchmark-environment fingerprints differ.

In [ ]:
COMPUTE_METRIC_DICTIONARY=pd.DataFrame([
 ["total_train_seconds","Optimization-only accumulated epoch wall time"],
 ["total_validation_selection_seconds","Validation/checkpoint-selection wall time"],
 ["best_cumulative_fit_seconds","Wall time through the validation-selected best epoch"],
 ["test_k_eval_seconds",f"Full test evaluation at K_eval={FIXED['k_eval']}"],
 ["test_k_eval_examples_per_sec","End-to-end test evaluation throughput"],
 ["k1_full_density_latency_ms_median","Warm repeated device-resident full K=1 density log-weight batch latency; RNG generation and host transfer excluded"],
 ["parameters / trainable_parameters","Model parameter counts"],
 ["parameter_bytes / best_checkpoint_bytes","In-memory parameters and serialized best checkpoint size"],
 ["train/validation/test peak GPU allocated bytes","Phases measured separately"],
 ["train/validation/test peak RSS bytes","Process+children RSS, sampled; practical approximation"],
 ["forward_linear_macs_per_example","Analytical dense-linear MAC proxy; excludes activation/likelihood/backward operations"],
 ["forward_linear_macs_per_training_epoch","Per-example forward linear MAC proxy × training examples"],
 ["forward_linear_macs_full_epoch_budget","Per-epoch forward linear MAC proxy × frozen epoch budget"],
 ["forward_linear_flops_per_example_approx","2× linear MACs; explicitly not claimed as exact total FLOPs"],
],columns=["metric","definition"])
display(COMPUTE_METRIC_DICTIONARY); COMPUTE_METRIC_DICTIONARY.to_csv(EXPORT_ROOT/"compute_metric_dictionary.csv",index=False)

,metric,definition
0,total_train_seconds,Optimization-only accumulated epoch wall time
1,total_validation_selection_seconds,Validation/checkpoint-selection wall time
2,best_cumulative_fit_seconds,Wall time through the validation-selected best...
3,test_k_eval_seconds,Full test evaluation at K_eval=1000
4,test_k_eval_examples_per_sec,End-to-end test evaluation throughput
5,k1_full_density_latency_ms_median,Warm repeated device-resident full K=1 density...
6,parameters / trainable_parameters,Model parameter counts
7,parameter_bytes / best_checkpoint_bytes,In-memory parameters and serialized best check...
8,train/validation/test peak GPU allocated bytes,Phases measured separately
9,train/validation/test peak RSS bytes,"Process+children RSS, sampled; practical appro..."


# 12. Results Aggregation

The EMNIST phase is not released merely because MNIST training finished: the complete MNIST grid must first undergo Phase-12/13 aggregation and bootstrap analysis. Whole missing configurations are detected against the expected manifest. Debug subsets are never promoted to final findings.

In [ ]:
def flatten_result(r):
    s=r["spec"]
    row={"experiment_id":r["experiment_id"],**s,"status":r["status"],"software_fingerprint":r["software_fingerprint"],
         "benchmark_environment_fingerprint":r["benchmark_environment_fingerprint"],"dataset_fingerprint":r["dataset_fingerprint"]}
    keys=["best_epoch","best_val_selection_bpd","best_cumulative_fit_seconds","total_train_seconds","total_validation_selection_seconds",
          "total_fit_selection_seconds","parameters","trainable_parameters","parameter_bytes","best_checkpoint_bytes","forward_linear_macs_per_example",
          "forward_linear_flops_per_example_approx","forward_linear_macs_per_training_epoch","forward_linear_macs_full_epoch_budget","k1_full_density_latency_ms_mean","k1_full_density_latency_ms_median","k1_full_density_latency_ms_p95",
          "k1_full_density_examples_per_sec","train_peak_gpu_allocated_bytes","validation_peak_gpu_allocated_bytes","train_peak_gpu_reserved_bytes","validation_peak_gpu_reserved_bytes","train_peak_rss_bytes","validation_peak_rss_bytes"]
    for k in keys: row[k]=r.get(k)
    for prefix in ["val_k_eval","test_k_eval"]:
        for k in ["negative_iwae_bound_nats_per_image","bpd_from_Lk","reconstruction_bpd_diagnostic","analytic_kl_bpd_diagnostic","active_units","seconds","examples_per_sec","peak_rss_bytes","peak_gpu_allocated_bytes","peak_gpu_reserved_bytes"]:
            row[f"{prefix}_{k}"]=r.get(f"{prefix}_{k}")
    return row


def bootstrap_mean_ci(values,reps=None,seed=None):
    a=np.asarray(values,float); reps=int(reps or FIXED["bootstrap_repetitions"]); rng=np.random.default_rng(FIXED["bootstrap_seed"] if seed is None else seed)
    draws=rng.choice(a,(reps,len(a)),replace=True).mean(1); return float(a.mean()),float(np.quantile(draws,.025)),float(np.quantile(draws,.975))

def bootstrap_paired_ci(diffs,label): return bootstrap_mean_ci(diffs,seed=stable_seed("bootstrap",FIXED["bootstrap_seed"],label))


def configuration_summary(df):
    rows=[]
    for keys,g in df.groupby(["dataset","split_name","resolution","k_train","method"],dropna=False):
        d,k,r,kk,m=keys; test=g["test_k_eval_bpd_from_Lk"].to_numpy(); val=g["val_k_eval_bpd_from_Lk"].to_numpy()
        tm,tlo,thi=bootstrap_mean_ci(test,seed=stable_seed("cfgtest",*keys)); vm,vlo,vhi=bootstrap_mean_ci(val,seed=stable_seed("cfgval",*keys))
        rows.append({"dataset":d,"split_name":k,"resolution":r,"k_train":kk,"method":m,"n_seeds":len(g),
                     "test_bpd_mean":tm,"test_bpd_sd":float(np.std(test,ddof=1)) if len(test)>1 else np.nan,"test_bpd_ci_low":tlo,"test_bpd_ci_high":thi,
                     "val_bpd_k_eval_mean":vm,"val_bpd_k_eval_sd":float(np.std(val,ddof=1)) if len(val)>1 else np.nan,"val_bpd_ci_low":vlo,"val_bpd_ci_high":vhi,
                     "reconstruction_bpd_mean":g["test_k_eval_reconstruction_bpd_diagnostic"].mean(),"analytic_kl_bpd_mean":g["test_k_eval_analytic_kl_bpd_diagnostic"].mean(),
                     "active_units_mean":g["test_k_eval_active_units"].mean(),"train_seconds_mean":g["total_train_seconds"].mean(),
                     "time_to_best_seconds_mean":g["best_cumulative_fit_seconds"].mean(),"best_epoch_mean":g["best_epoch"].mean(),
                     "parameters":g["parameters"].iloc[0],"forward_linear_macs_per_example":g["forward_linear_macs_per_example"].iloc[0]})
    return pd.DataFrame(rows)


def dreg_iwae_pairs(df,dataset):
    sub=df[(df.dataset==dataset)&(df.k_train>1)]; rows=[]
    for (r,k),g in sub.groupby(["resolution","k_train"]):
        piv=g.pivot(index="seed",columns="method",values="test_k_eval_bpd_from_Lk").dropna()
        if {"iwae","dreg"}.issubset(piv.columns):
            diff=(piv["dreg"]-piv["iwae"]).to_numpy(); mean,lo,hi=bootstrap_paired_ci(diff,f"dreg-{dataset}-{r}-{k}")
            rows.append({"dataset":dataset,"resolution":r,"k_train":k,"n":len(diff),"dreg_minus_iwae_bpd":mean,"ci_low":lo,"ci_high":hi})
    return pd.DataFrame(rows)



def dreg_iwae_convergence_pairs(df,dataset):
    rows=[]
    for (r,k),g in df[(df.dataset==dataset)&(df.k_train>1)].groupby(["resolution","k_train"]):
        pv=g.pivot(index="seed",columns="method",values=["best_epoch","best_cumulative_fit_seconds"])
        if len(pv) and all((metric,m) in pv.columns for metric in ["best_epoch","best_cumulative_fit_seconds"] for m in ["iwae","dreg"]):
            de=(pv[("best_epoch","dreg")]-pv[("best_epoch","iwae")]).dropna().to_numpy(float)
            dt=(pv[("best_cumulative_fit_seconds","dreg")]-pv[("best_cumulative_fit_seconds","iwae")]).dropna().to_numpy(float)
            if len(de) and len(dt):
                em,elo,ehi=bootstrap_paired_ci(de,f"conv-epoch-{dataset}-{r}-{k}")
                tm,tlo,thi=bootstrap_paired_ci(dt,f"conv-time-{dataset}-{r}-{k}")
                rows.append({"dataset":dataset,"resolution":r,"k_train":k,"n":min(len(de),len(dt)),
                             "dreg_minus_iwae_best_epoch":em,"best_epoch_ci_low":elo,"best_epoch_ci_high":ehi,
                             "dreg_minus_iwae_time_to_best_seconds":tm,"time_ci_low":tlo,"time_ci_high":thi})
    return pd.DataFrame(rows)


def adjacent_k_gains(df,dataset):
    rows=[]
    ks=sorted(EXPERIMENTAL["k_train_values"])
    for r in EXPERIMENTAL["observation_resolutions"]:
      for method in EXPERIMENTAL["estimators_k_gt_1"]:
        for klo,khi in zip(ks[:-1],ks[1:]):
            lo_method="vae" if klo==1 else method; hi_method="vae" if khi==1 else method
            a=df[(df.dataset==dataset)&(df.resolution==r)&(df.k_train==klo)&(df.method==lo_method)][["seed","test_k_eval_bpd_from_Lk"]]
            b=df[(df.dataset==dataset)&(df.resolution==r)&(df.k_train==khi)&(df.method==hi_method)][["seed","test_k_eval_bpd_from_Lk"]]
            p=a.merge(b,on="seed",suffixes=("_lo","_hi")); gain=(p.test_k_eval_bpd_from_Lk_lo-p.test_k_eval_bpd_from_Lk_hi).to_numpy()
            if len(gain):
                mean,ci0,ci1=bootstrap_paired_ci(gain,f"kgain-{dataset}-{r}-{method}-{klo}-{khi}")
                rows.append({"dataset":dataset,"resolution":r,"method_branch":method,"k_low":klo,"k_high":khi,"bpd_improvement":mean,"ci_low":ci0,"ci_high":ci1})
    return pd.DataFrame(rows)



def adjacent_r_effects(df,dataset):
    """Paired resolution effects at fixed K/method; positive means higher R has higher (worse) bpd."""
    rows=[]; rs=sorted(EXPERIMENTAL["observation_resolutions"])
    branches=[(1,"vae")]+[(k,m) for k in EXPERIMENTAL["k_train_values"] if k>1 for m in EXPERIMENTAL["estimators_k_gt_1"]]
    for rlo,rhi in zip(rs[:-1],rs[1:]):
        for k,m in branches:
            a=df[(df.dataset==dataset)&(df.resolution==rlo)&(df.k_train==k)&(df.method==m)][["seed","test_k_eval_bpd_from_Lk"]]
            b=df[(df.dataset==dataset)&(df.resolution==rhi)&(df.k_train==k)&(df.method==m)][["seed","test_k_eval_bpd_from_Lk"]]
            p=a.merge(b,on="seed",suffixes=("_lo","_hi")); diff=(p.test_k_eval_bpd_from_Lk_hi-p.test_k_eval_bpd_from_Lk_lo).to_numpy()
            if len(diff):
                mean,lo,hi=bootstrap_paired_ci(diff,f"reffect-{dataset}-{rlo}-{rhi}-{k}-{m}")
                rows.append({"dataset":dataset,"r_low":rlo,"r_high":rhi,"k_train":k,"method":m,"higher_R_minus_lower_R_bpd":mean,"ci_low":lo,"ci_high":hi,"n":len(diff)})
    return pd.DataFrame(rows)


def three_way_contrasts(df,dataset):
    rows=[]; rs=sorted(EXPERIMENTAL["observation_resolutions"]); ks=[k for k in sorted(EXPERIMENTAL["k_train_values"]) if k>1]
    for rlo,rhi in zip(rs[:-1],rs[1:]):
      for klo,khi in zip(ks[:-1],ks[1:]):
        vals=[]
        for seed in EXPERIMENTAL["seeds"]:
            def mdiff(r,k):
                q=df[(df.dataset==dataset)&(df.resolution==r)&(df.k_train==k)&(df.seed==seed)].set_index("method")["test_k_eval_bpd_from_Lk"]
                return float(q["dreg"]-q["iwae"]) if {"dreg","iwae"}.issubset(q.index) else np.nan
            ds=[mdiff(rlo,klo),mdiff(rlo,khi),mdiff(rhi,klo),mdiff(rhi,khi)]
            if np.isfinite(ds).all(): vals.append((ds[3]-ds[2])-(ds[1]-ds[0]))
        if vals:
            mean,lo,hi=bootstrap_paired_ci(vals,f"3way-{dataset}-{rlo}-{rhi}-{klo}-{khi}")
            rows.append({"dataset":dataset,"r_low":rlo,"r_high":rhi,"k_low":klo,"k_high":khi,"three_way_contrast_bpd":mean,"ci_low":lo,"ci_high":hi,"n":len(vals)})
    return pd.DataFrame(rows)


def saturation_table(k_gain):
    rows=[]
    for (dataset,r,method),g in k_gain.groupby(["dataset","resolution","method_branch"]):
        g=g.sort_values("k_high"); initial=float(g.iloc[0].bpd_improvement); threshold=FIXED["saturation_relative_gain_fraction"]*abs(initial)
        sat=None; status="not_reached"
        if initial<=0: status="initial_gain_nonpositive"
        else:
            arr=g.reset_index(drop=True)
            for i,row in arr.iterrows():
                if (arr.loc[i:,"bpd_improvement"].abs()<=threshold).all(): sat=int(row.k_high); status="reached"; break
        rows.append({"dataset":dataset,"resolution":r,"method_branch":method,"initial_gain_bpd":initial,"threshold_bpd":threshold,"saturation_k":sat,"status":status})
    return pd.DataFrame(rows)

MAIN_DF=pd.DataFrame([flatten_result(r) for r in MAIN_ROWS])
if len(MAIN_DF): MAIN_DF.to_csv(EXPORT_ROOT/"mnist_run_results.csv",index=False)
DEBUG_MODE=RUN_CONTROL["debug_max_runs"] is not None
if MAIN_COMPLETE and not DEBUG_MODE:
    MAIN_CONFIG_SUMMARY=configuration_summary(MAIN_DF); MAIN_PAIRWISE=dreg_iwae_pairs(MAIN_DF,"mnist"); MAIN_CONVERGENCE_PAIRS=dreg_iwae_convergence_pairs(MAIN_DF,"mnist"); MAIN_K_GAINS=adjacent_k_gains(MAIN_DF,"mnist"); MAIN_R_EFFECTS=adjacent_r_effects(MAIN_DF,"mnist"); MAIN_3WAY=three_way_contrasts(MAIN_DF,"mnist"); MAIN_SATURATION=saturation_table(MAIN_K_GAINS)
    MAIN_CONFIG_SUMMARY.to_csv(EXPORT_ROOT/"mnist_configuration_summary.csv",index=False); MAIN_PAIRWISE.to_csv(EXPORT_ROOT/"mnist_dreg_iwae_bootstrap.csv",index=False); MAIN_CONVERGENCE_PAIRS.to_csv(EXPORT_ROOT/"mnist_dreg_iwae_convergence_bootstrap.csv",index=False)
    MAIN_K_GAINS.to_csv(EXPORT_ROOT/"mnist_k_adjacent_gains.csv",index=False); MAIN_R_EFFECTS.to_csv(EXPORT_ROOT/"mnist_resolution_effects.csv",index=False); MAIN_3WAY.to_csv(EXPORT_ROOT/"mnist_three_way_contrasts.csv",index=False); MAIN_SATURATION.to_csv(EXPORT_ROOT/"mnist_saturation.csv",index=False)
    MNIST_PHASE13_COMPLETE=True
else:
    MNIST_PHASE13_COMPLETE=False
    print("Final MNIST Phase-13 analysis withheld: complete grid required and debug mode must be off.")

# 13. Statistical Analysis

No factorial ANOVA or small-sample t-test is introduced. The notebook uses mean±SD plus predeclared paired/nonparametric bootstrap CIs. Interaction contrasts are descriptive. With only three seeds, these intervals are intentionally interpreted as exploratory uncertainty summaries, not strong confirmatory inference.

In [ ]:
if MNIST_PHASE13_COMPLETE:
    display(MAIN_CONFIG_SUMMARY); display(MAIN_R_EFFECTS); display(MAIN_PAIRWISE); display(MAIN_CONVERGENCE_PAIRS); display(MAIN_3WAY); display(MAIN_SATURATION)
    MAIN_ANALYSIS_GATE={"phase13_complete":True,"main_complete":True,"source_spec_sha256":SOURCE_SPEC_SHA256,"training_protocol_fingerprint":TRAINING_PROTOCOL_FINGERPRINT}
    atomic_json(MAIN_ANALYSIS_GATE,EXPORT_ROOT/"mnist_phase13_gate.json")
else:
    MAIN_ANALYSIS_GATE={"phase13_complete":False,"main_complete":MAIN_COMPLETE,"debug_mode":DEBUG_MODE,"missing":MAIN_MISSING}
    atomic_json(MAIN_ANALYSIS_GATE,EXPORT_ROOT/"mnist_phase13_gate.json")

,dataset,split_name,resolution,k_train,method,n_seeds,test_bpd_mean,test_bpd_sd,test_bpd_ci_low,test_bpd_ci_high,...,val_bpd_ci_low,val_bpd_ci_high,reconstruction_bpd_mean,analytic_kl_bpd_mean,active_units_mean,train_seconds_mean,time_to_best_seconds_mean,best_epoch_mean,parameters,forward_linear_macs_per_example
0,mnist,official,2,1,vae,3,0.139614,0.001342,0.138127,0.140735,...,0.139265,0.141501,0.105955,0.046352,15.000000,201.868729,218.292041,97.333333,407224,405600
1,mnist,official,2,5,dreg,3,0.133345,0.000126,0.133221,0.133473,...,0.134178,0.134659,0.105202,0.047428,20.000000,236.788558,257.265602,98.000000,407224,1208800
2,mnist,official,2,5,iwae,3,0.134733,0.000163,0.134632,0.134921,...,0.135727,0.136235,0.109367,0.045567,18.666667,217.078847,233.156036,96.000000,407224,1208800
3,mnist,official,2,10,dreg,3,0.132677,0.000043,0.132642,0.132726,...,0.133669,0.133853,0.109915,0.045652,20.000000,241.048456,260.109607,97.333333,407224,2212800
4,mnist,official,2,10,iwae,3,0.134086,0.000096,0.133979,0.134164,...,0.135108,0.135232,0.112677,0.045175,20.000000,218.436481,227.099026,93.000000,407224,2212800
5,mnist,official,2,20,dreg,3,0.132689,0.000178,0.132485,0.132816,...,0.133392,0.133909,0.114895,0.044181,20.000000,238.198741,252.230607,95.666667,407224,4220800
6,mnist,official,2,20,iwae,3,0.133663,0.000353,0.133346,0.134044,...,0.134373,0.135152,0.117330,0.043507,20.000000,219.344046,227.162133,92.333333,407224,4220800
7,mnist,official,2,50,dreg,3,0.132384,0.000027,0.132359,0.132413,...,0.133357,0.133479,0.120727,0.042993,20.000000,239.032446,263.804509,98.000000,407224,10244800
8,mnist,official,2,50,iwae,3,0.133384,0.000108,0.133279,0.133495,...,0.134231,0.134516,0.125926,0.041418,20.000000,228.717932,243.050223,95.666667,407224,10244800
9,mnist,official,16,1,vae,3,0.614582,0.001545,0.613159,0.616226,...,0.617280,0.620329,0.568439,0.067734,18.000000,219.899468,237.094748,98.000000,2770984,2757600


,dataset,r_low,r_high,k_train,method,higher_R_minus_lower_R_bpd,ci_low,ci_high,n
0,mnist,2,16,1,vae,0.474968,0.473624,0.476247,3
1,mnist,2,16,5,iwae,0.472031,0.471256,0.473446,3
2,mnist,2,16,5,dreg,0.471735,0.471296,0.472211,3
3,mnist,2,16,10,iwae,0.471682,0.471199,0.472187,3
4,mnist,2,16,10,dreg,0.471171,0.470796,0.471876,3
5,mnist,2,16,20,iwae,0.471373,0.471192,0.471664,3
6,mnist,2,16,20,dreg,0.470489,0.470071,0.471102,3
7,mnist,2,16,50,iwae,0.471108,0.470538,0.471975,3
8,mnist,2,16,50,dreg,0.470455,0.470041,0.471094,3
9,mnist,16,256,1,vae,1.010351,1.009021,1.012509,3


,dataset,resolution,k_train,n,dreg_minus_iwae_bpd,ci_low,ci_high
0,mnist,2,5,3,-0.001388,-0.001448,-0.001307
1,mnist,2,10,3,-0.001410,-0.001453,-0.001337
2,mnist,2,20,3,-0.000974,-0.001279,-0.000782
3,mnist,2,50,3,-0.000999,-0.001082,-0.000920
4,mnist,16,5,3,-0.001685,-0.003560,-0.000352
5,mnist,16,10,3,-0.001920,-0.002783,-0.000661
6,mnist,16,20,3,-0.001858,-0.002471,-0.000872
7,mnist,16,50,3,-0.001652,-0.002742,-0.000637
8,mnist,256,5,3,-0.001453,-0.004510,0.004402
9,mnist,256,10,3,0.001510,0.000609,0.002183


,dataset,resolution,k_train,n,dreg_minus_iwae_best_epoch,best_epoch_ci_low,best_epoch_ci_high,dreg_minus_iwae_time_to_best_seconds,time_ci_low,time_ci_high
0,mnist,2,5,3,2.000000,0.0,6.0,24.109566,18.610606,29.584179
1,mnist,2,10,3,4.333333,2.0,8.0,33.010581,24.561475,47.171874
2,mnist,2,20,3,3.333333,-5.0,13.0,25.068474,12.836971,46.136035
3,mnist,2,50,3,2.333333,0.0,4.0,20.754285,-2.147477,44.485870
4,mnist,16,5,3,3.000000,0.0,6.0,41.376462,26.386981,53.430324
5,mnist,16,10,3,1.000000,-1.0,2.0,45.996772,41.298626,53.828676
6,mnist,16,20,3,-0.333333,-4.0,4.0,122.904427,110.978088,138.528921
7,mnist,16,50,3,0.000000,-1.0,2.0,333.554489,320.601869,356.160439
8,mnist,256,5,3,1.333333,1.0,2.0,45.605935,43.089038,50.135247
9,mnist,256,10,3,-1.000000,-2.0,0.0,38.961122,33.938601,42.882660


,dataset,r_low,r_high,k_low,k_high,three_way_contrast_bpd,ci_low,ci_high,n
0,mnist,2,16,5,10,-0.000214,-0.002300,0.001287,3
1,mnist,2,16,10,20,-0.000373,-0.002045,0.000773,3
2,mnist,2,16,20,50,0.000230,-0.000406,0.001476,3
3,mnist,16,256,5,10,0.003199,-0.003909,0.009124,3
4,mnist,16,256,10,20,-0.005596,-0.008307,-0.002682,3
5,mnist,16,256,20,50,0.003261,0.000761,0.006013,3


,dataset,resolution,method_branch,initial_gain_bpd,threshold_bpd,saturation_k,status
0,mnist,2,dreg,0.006269,0.000627,20.0,reached
1,mnist,2,iwae,0.004880,0.000488,20.0,reached
2,mnist,16,dreg,0.009502,0.000950,20.0,reached
3,mnist,16,iwae,0.007817,0.000782,20.0,reached
4,mnist,256,dreg,0.027123,0.002712,NaN,not_reached
5,mnist,256,iwae,0.025670,0.002567,NaN,not_reached


## Phase 16 — EMNIST matched generalization

The EMNIST grid is released **only after** the complete MNIST Phase-13 analysis exists. It inherits the same architecture, optimizer, LR, epoch budget, R/K/estimator grid, and seeds. Absolute task bpd is not treated as the generalization claim; matched K-gain and DReG-vs-IWAE patterns are compared directly.

In [ ]:
EMNIST_EXECUTED=[]
if PHASES["run_emnist_generalization_grid"]:
    if not MNIST_PHASE13_COMPLETE:
        print("EMNIST withheld: complete MNIST Phase-13 analysis is required first.")
    else:
        if "emnist" not in DATASETS:
            DATASETS["emnist"]=load_dataset_bundle("emnist")
            b=DATASETS["emnist"]
            np.save(SPLIT_DIR / f"emnist_{b.split_name}_train_indices.npy", b.train_indices.numpy())
            np.save(SPLIT_DIR / f"emnist_{b.split_name}_val_indices.npy", b.val_indices.numpy())
            with open(SPLIT_DIR / f"emnist_{b.split_name}_provenance.json", "w") as f:
                json.dump({**b.provenance,"dataset_fingerprint":b.dataset_fingerprint},f,indent=2)
            train_f=b.train_images.float().div(255.0); val_f=b.val_images.float().div(255.0)
            em_stats=pd.DataFrame([{
                "dataset":"emnist","split":b.split_name,"train_n":len(b.train_images),"val_n":len(b.val_images),"test_n_sealed":len(b.test_images),
                "train_pixel_mean":float(train_f.mean()),"train_pixel_std":float(train_f.std()),"val_pixel_mean":float(val_f.mean()),"val_pixel_std":float(val_f.std()),
                "train_classes":int(torch.unique(b.train_labels).numel()),"dataset_fingerprint":b.dataset_fingerprint}])
            em_stats.to_csv(EXPORT_ROOT/"emnist_dataset_statistics_phase16.csv",index=False); display(em_stats)
            em_q=pd.DataFrame(quantization_summary(b)); em_q.to_csv(EXPORT_ROOT/"emnist_quantization_statistics_phase16.csv",index=False); display(em_q)
        EMNIST_EXECUTED=execute_specs(EMNIST_SPECS,collect_pilot_gradients=False)
EMNIST_ROWS,EMNIST_MISSING=collect_completed(EMNIST_SPECS) if "emnist" in DATASETS else ([],[s.experiment_id() for s in EMNIST_SPECS])
EMNIST_COMPLETE=(len(EMNIST_MISSING)==0)
pd.DataFrame({"missing_experiment_id":EMNIST_MISSING}).to_csv(EXPORT_ROOT/"emnist_missing_runs.csv",index=False)
print(f"EMNIST complete: {EMNIST_COMPLETE}; completed={len(EMNIST_ROWS)}/{len(EMNIST_SPECS)}")

,dataset,split,train_n,val_n,test_n_sealed,train_pixel_mean,train_pixel_std,val_pixel_mean,val_pixel_std,train_classes,dataset_fingerprint
0,emnist,letters,104000,20800,20800,0.172131,0.330866,0.17271,0.331351,26,4aeaacc1b0169435b4fd6825685990a7e80ae8966395ac...


,dataset,split,R,observed_levels,min_level,max_level,mean_normalized
0,emnist,letters,2,2,0,1,0.166406
1,emnist,letters,16,16,0,15,0.171685
2,emnist,letters,256,256,0,255,0.171862


REUSE generalization_emnist-letters_r2_vae_k1_z20_seed1 from /home/imohammedali/projects/pgm/latent_density_results/research_grade_z20_v2/experiments/generalization_emnist-letters_r2_vae_k1_z20_seed1/attempt_20260906T120338_637707Z
REUSE generalization_emnist-letters_r2_vae_k1_z20_seed2 from /home/imohammedali/projects/pgm/latent_density_results/research_grade_z20_v2/experiments/generalization_emnist-letters_r2_vae_k1_z20_seed2/attempt_20260906T121122_035585Z
REUSE generalization_emnist-letters_r2_vae_k1_z20_seed3 from /home/imohammedali/projects/pgm/latent_density_results/research_grade_z20_v2/experiments/generalization_emnist-letters_r2_vae_k1_z20_seed3/attempt_20260906T121908_418773Z
REUSE generalization_emnist-letters_r2_iwae_k5_z20_seed1 from /home/imohammedali/projects/pgm/latent_density_results/research_grade_z20_v2/experiments/generalization_emnist-letters_r2_iwae_k5_z20_seed1/attempt_20260906T122647_901078Z
REUSE generalization_emnist-letters_r2_iwae_k5_z20_seed2 from /home/im

In [ ]:
EMNIST_DF=pd.DataFrame([flatten_result(r) for r in EMNIST_ROWS])
if len(EMNIST_DF): EMNIST_DF.to_csv(EXPORT_ROOT/"emnist_run_results.csv",index=False)
ALL_DF=pd.concat([x for x in [MAIN_DF,EMNIST_DF] if len(x)],ignore_index=True) if (len(MAIN_DF) or len(EMNIST_DF)) else pd.DataFrame()
FINAL_COMPLETE=MAIN_COMPLETE and (not PHASES["run_emnist_generalization_grid"] or EMNIST_COMPLETE) and not DEBUG_MODE
if EMNIST_COMPLETE and not DEBUG_MODE:
    EMNIST_CONFIG_SUMMARY=configuration_summary(EMNIST_DF); EMNIST_PAIRWISE=dreg_iwae_pairs(EMNIST_DF,"emnist"); EMNIST_CONVERGENCE_PAIRS=dreg_iwae_convergence_pairs(EMNIST_DF,"emnist"); EMNIST_K_GAINS=adjacent_k_gains(EMNIST_DF,"emnist"); EMNIST_R_EFFECTS=adjacent_r_effects(EMNIST_DF,"emnist"); EMNIST_3WAY=three_way_contrasts(EMNIST_DF,"emnist"); EMNIST_SATURATION=saturation_table(EMNIST_K_GAINS)
    for name,obj in [("emnist_configuration_summary",EMNIST_CONFIG_SUMMARY),("emnist_dreg_iwae_bootstrap",EMNIST_PAIRWISE),("emnist_dreg_iwae_convergence_bootstrap",EMNIST_CONVERGENCE_PAIRS),("emnist_k_adjacent_gains",EMNIST_K_GAINS),("emnist_resolution_effects",EMNIST_R_EFFECTS),("emnist_three_way_contrasts",EMNIST_3WAY),("emnist_saturation",EMNIST_SATURATION)]: obj.to_csv(EXPORT_ROOT/f"{name}.csv",index=False)
    RQ5_DREG=MAIN_PAIRWISE.merge(EMNIST_PAIRWISE,on=["resolution","k_train"],suffixes=("_mnist","_emnist"))
    RQ5_K=MAIN_K_GAINS.merge(EMNIST_K_GAINS,on=["resolution","method_branch","k_low","k_high"],suffixes=("_mnist","_emnist"))
    RQ5_R=MAIN_R_EFFECTS.merge(EMNIST_R_EFFECTS,on=["r_low","r_high","k_train","method"],suffixes=("_mnist","_emnist"))
    RQ5_3WAY=MAIN_3WAY.merge(EMNIST_3WAY,on=["r_low","r_high","k_low","k_high"],suffixes=("_mnist","_emnist"))
    RQ5_SAT=MAIN_SATURATION.merge(EMNIST_SATURATION,on=["resolution","method_branch"],suffixes=("_mnist","_emnist"))
    RQ5_DREG.to_csv(EXPORT_ROOT/"rq5_matched_dreg_effects.csv",index=False); RQ5_K.to_csv(EXPORT_ROOT/"rq5_matched_k_gains.csv",index=False); RQ5_R.to_csv(EXPORT_ROOT/"rq5_matched_resolution_effects.csv",index=False); RQ5_3WAY.to_csv(EXPORT_ROOT/"rq5_matched_three_way_contrasts.csv",index=False); RQ5_SAT.to_csv(EXPORT_ROOT/"rq5_matched_saturation.csv",index=False)

# 14. Visualization

Primary comparative figures use bootstrap 95% CIs. Convergence figures cover every configured R/K/estimator branch; they are diagnostic rather than significance plots.

In [ ]:
PLOT_DIR=EXPORT_ROOT/"plots"; PLOT_DIR.mkdir(parents=True,exist_ok=True)
def plot_config_curves(summary,dataset):
    if summary is None or summary.empty: return
    for r in sorted(summary.resolution.unique()):
        fig,ax=plt.subplots(figsize=(7,4.5)); sub=summary[summary.resolution==r]
        for method in ["iwae","dreg"]:
            rows=[]
            base=sub[(sub.k_train==1)&(sub.method=="vae")]
            if len(base): rows.append(base.iloc[0])
            rows += [x for _,x in sub[(sub.k_train>1)&(sub.method==method)].sort_values("k_train").iterrows()]
            if not rows: continue
            x=np.array([z.k_train for z in rows]); y=np.array([z.test_bpd_mean for z in rows]); lo=np.array([z.test_bpd_ci_low for z in rows]); hi=np.array([z.test_bpd_ci_high for z in rows])
            ax.errorbar(x,y,yerr=np.vstack([y-lo,hi-y]),marker="o",capsize=3,label=method.upper())
        ax.set(xlabel="K_train",ylabel=f"Test bpd @ K_eval={FIXED['k_eval']}",title=f"{dataset.upper()} R={r}: performance with bootstrap CI"); ax.legend(); fig.tight_layout(); fig.savefig(PLOT_DIR/f"{dataset}_r{r}_performance_bootstrap.png",dpi=160); plt.close(fig)


def plot_effect_ci(table,dataset,value_col,title_suffix,file_suffix):
    if table is None or table.empty: return
    fig,ax=plt.subplots(figsize=(7,4.5))
    for r,g in table.groupby("resolution"):
        g=g.sort_values("k_train"); y=g[value_col].to_numpy(); lo=g["ci_low"].to_numpy(); hi=g["ci_high"].to_numpy()
        ax.errorbar(g["k_train"],y,yerr=np.vstack([y-lo,hi-y]),marker="o",capsize=3,label=f"R={r}")
    ax.axhline(0,linewidth=1); ax.set(xlabel="K_train",ylabel=value_col,title=f"{dataset.upper()}: {title_suffix}"); ax.legend(); fig.tight_layout(); fig.savefig(PLOT_DIR/f"{dataset}_{file_suffix}.png",dpi=160); plt.close(fig)


def load_history_for_result(row):
    p=Path(row.get("artifact_dir",row["run_dir"]))/"history.csv"
    return pd.read_csv(p) if p.exists() else None

def plot_all_convergence(rows,dataset):
    rdf=[r for r in rows if r["spec"]["dataset"]==dataset]
    for r in EXPERIMENTAL["observation_resolutions"]:
        fig,ax=plt.subplots(figsize=(8,5))
        branches=[(1,"vae")]+[(k,m) for k in EXPERIMENTAL["k_train_values"] if k>1 for m in EXPERIMENTAL["estimators_k_gt_1"]]
        for k,m in branches:
            hs=[]
            for rr in rdf:
                s=rr["spec"]
                if s["resolution"]==r and s["k_train"]==k and s["method"]==m:
                    h=load_history_for_result(rr)
                    if h is not None: hs.append(h["val_selection_bpd"].to_numpy())
            if hs and len({len(x) for x in hs})==1:
                a=np.stack(hs); ax.plot(np.arange(1,a.shape[1]+1),a.mean(0),label=f"K{k}-{m}")
        ax.set(xlabel="Epoch",ylabel="Validation selection bpd",title=f"{dataset.upper()} R={r}: all convergence branches"); ax.legend(fontsize=7,ncol=2); fig.tight_layout(); fig.savefig(PLOT_DIR/f"{dataset}_r{r}_all_convergence.png",dpi=160); plt.close(fig)

if MNIST_PHASE13_COMPLETE:
    plot_config_curves(MAIN_CONFIG_SUMMARY,"mnist"); plot_effect_ci(MAIN_PAIRWISE,"mnist","dreg_minus_iwae_bpd","DReG − IWAE bpd with paired bootstrap CI","dreg_minus_iwae_bootstrap"); plot_all_convergence(MAIN_ROWS,"mnist")
if EMNIST_COMPLETE and not DEBUG_MODE:
    plot_config_curves(EMNIST_CONFIG_SUMMARY,"emnist"); plot_effect_ci(EMNIST_PAIRWISE,"emnist","dreg_minus_iwae_bpd","DReG − IWAE bpd with paired bootstrap CI","dreg_minus_iwae_bootstrap"); plot_all_convergence(EMNIST_ROWS,"emnist")

# 15. Final Comparison

The final tables answer the five predeclared result groups: observation resolution, K behavior/saturation, IWAE vs DReG, interaction patterns, and efficiency. Comparative efficiency is emitted only when all compared runs share one benchmark-environment fingerprint.

In [ ]:
def efficiency_allowed(df):
    if df.empty: return False
    return (not RUN_CONTROL["require_uniform_benchmark_environment_for_efficiency"]) or df["benchmark_environment_fingerprint"].nunique()==1

FINAL_TABLES={}
if PHASES["run_final_analysis"] and FINAL_COMPLETE:
    FINAL_TABLES["mnist_configuration"]=MAIN_CONFIG_SUMMARY
    FINAL_TABLES["mnist_resolution_effects"]=MAIN_R_EFFECTS
    FINAL_TABLES["mnist_dreg_iwae"]=MAIN_PAIRWISE
    FINAL_TABLES["mnist_convergence_speed"]=MAIN_CONVERGENCE_PAIRS
    FINAL_TABLES["mnist_k_gains"]=MAIN_K_GAINS
    FINAL_TABLES["mnist_saturation"]=MAIN_SATURATION
    FINAL_TABLES["mnist_three_way"]=MAIN_3WAY
    if EMNIST_COMPLETE:
        FINAL_TABLES.update({"emnist_configuration":EMNIST_CONFIG_SUMMARY,"emnist_resolution_effects":EMNIST_R_EFFECTS,"emnist_dreg_iwae":EMNIST_PAIRWISE,"emnist_convergence_speed":EMNIST_CONVERGENCE_PAIRS,"emnist_k_gains":EMNIST_K_GAINS,"emnist_saturation":EMNIST_SATURATION,"emnist_three_way":EMNIST_3WAY})
    if efficiency_allowed(ALL_DF):
        EFFICIENCY=ALL_DF.groupby(["dataset","resolution","k_train","method"],as_index=False).agg(
            test_bpd=("test_k_eval_bpd_from_Lk","mean"),train_seconds=("total_train_seconds","mean"),time_to_best=("best_cumulative_fit_seconds","mean"),
            test_throughput=("test_k_eval_examples_per_sec","mean"),k1_latency_ms=("k1_full_density_latency_ms_median","mean"),parameters=("parameters","first"),
            train_peak_gpu_bytes=("train_peak_gpu_allocated_bytes","max"),forward_macs=("forward_linear_macs_per_example","first"))
        EFFICIENCY.to_csv(EXPORT_ROOT/"efficiency_comparison.csv",index=False); FINAL_TABLES["efficiency"]=EFFICIENCY
    else: print("Efficiency comparison suppressed because benchmark environments are not uniform.")
    for name,t in FINAL_TABLES.items(): print("\n",name); display(t)
else:
    print("FINAL SCIENTIFIC COMPARISON WITHHELD: complete requested grids and non-debug execution are required.")


 mnist_configuration


,dataset,split_name,resolution,k_train,method,n_seeds,test_bpd_mean,test_bpd_sd,test_bpd_ci_low,test_bpd_ci_high,...,val_bpd_ci_low,val_bpd_ci_high,reconstruction_bpd_mean,analytic_kl_bpd_mean,active_units_mean,train_seconds_mean,time_to_best_seconds_mean,best_epoch_mean,parameters,forward_linear_macs_per_example
0,mnist,official,2,1,vae,3,0.139614,0.001342,0.138127,0.140735,...,0.139265,0.141501,0.105955,0.046352,15.000000,201.868729,218.292041,97.333333,407224,405600
1,mnist,official,2,5,dreg,3,0.133345,0.000126,0.133221,0.133473,...,0.134178,0.134659,0.105202,0.047428,20.000000,236.788558,257.265602,98.000000,407224,1208800
2,mnist,official,2,5,iwae,3,0.134733,0.000163,0.134632,0.134921,...,0.135727,0.136235,0.109367,0.045567,18.666667,217.078847,233.156036,96.000000,407224,1208800
3,mnist,official,2,10,dreg,3,0.132677,0.000043,0.132642,0.132726,...,0.133669,0.133853,0.109915,0.045652,20.000000,241.048456,260.109607,97.333333,407224,2212800
4,mnist,official,2,10,iwae,3,0.134086,0.000096,0.133979,0.134164,...,0.135108,0.135232,0.112677,0.045175,20.000000,218.436481,227.099026,93.000000,407224,2212800
5,mnist,official,2,20,dreg,3,0.132689,0.000178,0.132485,0.132816,...,0.133392,0.133909,0.114895,0.044181,20.000000,238.198741,252.230607,95.666667,407224,4220800
6,mnist,official,2,20,iwae,3,0.133663,0.000353,0.133346,0.134044,...,0.134373,0.135152,0.117330,0.043507,20.000000,219.344046,227.162133,92.333333,407224,4220800
7,mnist,official,2,50,dreg,3,0.132384,0.000027,0.132359,0.132413,...,0.133357,0.133479,0.120727,0.042993,20.000000,239.032446,263.804509,98.000000,407224,10244800
8,mnist,official,2,50,iwae,3,0.133384,0.000108,0.133279,0.133495,...,0.134231,0.134516,0.125926,0.041418,20.000000,228.717932,243.050223,95.666667,407224,10244800
9,mnist,official,16,1,vae,3,0.614582,0.001545,0.613159,0.616226,...,0.617280,0.620329,0.568439,0.067734,18.000000,219.899468,237.094748,98.000000,2770984,2757600



 mnist_resolution_effects


,dataset,r_low,r_high,k_train,method,higher_R_minus_lower_R_bpd,ci_low,ci_high,n
0,mnist,2,16,1,vae,0.474968,0.473624,0.476247,3
1,mnist,2,16,5,iwae,0.472031,0.471256,0.473446,3
2,mnist,2,16,5,dreg,0.471735,0.471296,0.472211,3
3,mnist,2,16,10,iwae,0.471682,0.471199,0.472187,3
4,mnist,2,16,10,dreg,0.471171,0.470796,0.471876,3
5,mnist,2,16,20,iwae,0.471373,0.471192,0.471664,3
6,mnist,2,16,20,dreg,0.470489,0.470071,0.471102,3
7,mnist,2,16,50,iwae,0.471108,0.470538,0.471975,3
8,mnist,2,16,50,dreg,0.470455,0.470041,0.471094,3
9,mnist,16,256,1,vae,1.010351,1.009021,1.012509,3



 mnist_dreg_iwae


...

,dataset,resolution,method_branch,initial_gain_bpd,threshold_bpd,saturation_k,status
0,emnist,2,dreg,0.004064,0.000406,NaN,not_reached
1,emnist,2,iwae,0.002593,0.000259,NaN,not_reached
2,emnist,16,dreg,0.008702,0.000870,50.0,reached
3,emnist,16,iwae,0.007509,0.000751,NaN,not_reached
4,emnist,256,dreg,0.033790,0.003379,50.0,reached
5,emnist,256,iwae,0.029934,0.002993,20.0,reached



 emnist_three_way


,dataset,r_low,r_high,k_low,k_high,three_way_contrast_bpd,ci_low,ci_high,n
0,emnist,2,16,5,10,0.000673,-0.000179,0.002364,3
1,emnist,2,16,10,20,-0.002149,-0.002481,-0.001871,3
2,emnist,2,16,20,50,0.001050,0.000783,0.001306,3
3,emnist,16,256,5,10,0.000027,-0.004427,0.003030,3
4,emnist,16,256,10,20,-0.000346,-0.002684,0.003538,3
5,emnist,16,256,20,50,-0.000863,-0.003366,0.003849,3



 efficiency


,dataset,resolution,k_train,method,test_bpd,train_seconds,time_to_best,test_throughput,k1_latency_ms,parameters,train_peak_gpu_bytes,forward_macs
0,emnist,2,1,vae,0.173580,412.613915,450.356777,7617.387765,0.850524,407224,77174272,405600
1,emnist,2,5,dreg,0.169516,483.783837,524.372510,8085.679754,0.842303,407224,82048512,1208800
2,emnist,2,5,iwae,0.170987,444.551445,467.920009,7316.066297,0.852010,407224,82048512,1208800
3,emnist,2,10,dreg,0.168540,506.198928,553.518212,7449.355909,0.845487,407224,96742400,2212800
4,emnist,2,10,iwae,0.170163,444.533116,472.334862,7568.616963,1.004894,407224,90594304,2212800
5,emnist,2,20,dreg,0.168460,481.511197,513.325292,7467.801013,0.839839,407224,114660864,4220800
6,emnist,2,20,iwae,0.169470,440.594854,469.141144,7586.591623,1.056001,407224,106036224,4220800
7,emnist,2,50,dreg,0.167954,502.141170,526.248476,6671.684217,1.236031,407224,175138304,10244800
8,emnist,2,50,iwae,0.169137,439.478319,485.315597,7728.111237,0.838568,407224,153263104,10244800
9,emnist,16,1,vae,0.880444,458.008502,485.957944,2172.132031,0.798236,2770984,147606528,2757600


# 16. Best Configuration

A single configuration is selected **only from common-`K_eval` validation bpd averaged across seeds**. Its already-computed test result is then reported. The minimum test cell is never used as the selector.

In [ ]:
BEST_CONFIGURATIONS=[]
if PHASES["run_final_analysis"] and FINAL_COMPLETE:
    summaries=[MAIN_CONFIG_SUMMARY]+([EMNIST_CONFIG_SUMMARY] if EMNIST_COMPLETE else [])
    for summary in summaries:
        dataset=summary.dataset.iloc[0]; best=summary.sort_values(["val_bpd_k_eval_mean","resolution","k_train","method"]).iloc[0]
        BEST_CONFIGURATIONS.append({"dataset":dataset,"selection_basis":f"mean validation bpd @ K_eval={FIXED['k_eval']}","resolution":int(best.resolution),
                                    "k_train":int(best.k_train),"method":best.method,"validation_bpd_mean":float(best.val_bpd_k_eval_mean),
                                    "test_bpd_mean_after_selection":float(best.test_bpd_mean),"test_bpd_sd_after_selection":float(best.test_bpd_sd)})
BEST_CONFIGURATIONS=pd.DataFrame(BEST_CONFIGURATIONS)
if len(BEST_CONFIGURATIONS): display(BEST_CONFIGURATIONS); BEST_CONFIGURATIONS.to_csv(EXPORT_ROOT/"best_configurations_validation_selected.csv",index=False)

,dataset,selection_basis,resolution,k_train,method,validation_bpd_mean,test_bpd_mean_after_selection,test_bpd_sd_after_selection
0,mnist,mean validation bpd @ K_eval=1000,2,50,dreg,0.133425,0.132384,0.000027
1,emnist,mean validation bpd @ K_eval=1000,2,50,dreg,0.167620,0.167954,0.000423


## Scientific limitations that remain by design

- `R` is intentionally coupled to the prescribed observation likelihood. Therefore RQ1 estimates the effect of the **resolution/observation-model condition**, not a likelihood-family-invariant causal effect of resolution alone.
- The R=16 categorical head has substantially more parameters than the R=2 Bernoulli and R=256 two-parameter logistic heads. This is quantified in the exported architecture/compute tables and must be acknowledged when interpreting R effects; changing the heads to equalize capacity would violate the frozen protocol.
- `K_eval=1000` yields a finite-sample IWAE lower-bound estimate, not exact log likelihood. The notebook therefore names the stored quantity `negative_iwae_bound`/`bpd_from_Lk` rather than exact NLL.
- Three seeds per cell permit paired descriptive uncertainty summaries but weak confirmatory inference. Bootstrap intervals are consequently framed as exploratory, as predeclared in the source plan.
- The complete 81+81 grid is larger than many single Kaggle sessions. Sharding/resume solves execution management without changing the design, but comparative efficiency is only released for a uniform benchmark environment.

# 17. Save / Export Results & Final Self-Audit

All exports are session-versioned, while each experiment attempt has an immutable timestamped directory. The final audit manifest records grid completeness, fingerprints, protocol conformance, and artifact hashes. It does not claim experimental findings when required runs are absent.

In [ ]:
def sha256_file(path):
    h=hashlib.sha256()
    with open(path,"rb") as f:
        for chunk in iter(lambda:f.read(1<<20),b""): h.update(chunk)
    return h.hexdigest()

# Persist current configuration/environment separately from immutable per-run configs.
atomic_json({"EXPERIMENTAL":EXPERIMENTAL,"FIXED":FIXED,"PILOT":PILOT,"PHASES":PHASES,"RUN_CONTROL":RUN_CONTROL,"EMNIST_CONFIG":EMNIST_CONFIG,
             "source_spec_sha256":SOURCE_SPEC_SHA256,"training_implementation_fingerprint":TRAINING_IMPLEMENTATION_FINGERPRINT,
             "training_protocol_fingerprint":TRAINING_PROTOCOL_FINGERPRINT,"study_design_fingerprint":STUDY_DESIGN_FINGERPRINT},EXPORT_ROOT/"complete_configuration.json")
atomic_json(ENVIRONMENT,EXPORT_ROOT/"environment.json")

artifacts=[]
for p in EXPORT_ROOT.rglob("*"):
    if p.is_file() and p.name!="artifact_manifest.csv":
        try: artifacts.append({"relative_path":str(p.relative_to(EXPORT_ROOT)),"bytes":p.stat().st_size,"sha256":sha256_file(p)})
        except Exception: pass
ARTIFACT_MANIFEST=pd.DataFrame(artifacts); ARTIFACT_MANIFEST.to_csv(EXPORT_ROOT/"artifact_manifest.csv",index=False)

expected_main={s.experiment_id() for s in MNIST_SPECS}; observed_main={r["experiment_id"] for r in MAIN_ROWS}
expected_em={s.experiment_id() for s in EMNIST_SPECS}; observed_em={r["experiment_id"] for r in EMNIST_ROWS}
FINAL_AUDIT={
    "latent_dim_exactly_20":FIXED["latent_dim"]==20,
    "source_spec_sha256":SOURCE_SPEC_SHA256,
    "training_implementation_fingerprint":TRAINING_IMPLEMENTATION_FINGERPRINT,
    "training_protocol_fingerprint":TRAINING_PROTOCOL_FINGERPRINT,
    "study_design_fingerprint":STUDY_DESIGN_FINGERPRINT,
    "frozen_main_design_conforming":set(EXPERIMENTAL["observation_resolutions"])=={2,16,256} and set(EXPERIMENTAL["k_train_values"])=={1,5,10,20,50} and set(EXPERIMENTAL["estimators_k_gt_1"])=={"iwae","dreg"} and EXPERIMENTAL["seeds"]==[1,2,3],
    "latent_sensitivity_excluded_by_current_z20_constraint":True,
    "pilot_passed":bool(PILOT_PASSED),
    "mnist_grid_exactly_complete":expected_main==observed_main,
    "emnist_grid_exactly_complete":expected_em==observed_em if PHASES["run_emnist_generalization_grid"] else None,
    "debug_mode":DEBUG_MODE,
    "best_configuration_selection_uses_validation_only":True,
    "test_used_for_checkpoint_selection":False,
    "final_scientific_findings_released":bool(PHASES["run_final_analysis"] and FINAL_COMPLETE),
    "scientific_software_fingerprints":sorted(ALL_DF.software_fingerprint.unique().tolist()) if len(ALL_DF) else [],
    "benchmark_environment_fingerprints":sorted(ALL_DF.benchmark_environment_fingerprint.unique().tolist()) if len(ALL_DF) else [],
    "emnist_protocol_deviation":EMNIST_PROTOCOL_DEVIATION,
}
FINAL_AUDIT["verdict"]="PASS" if (FINAL_AUDIT["latent_dim_exactly_20"] and FINAL_AUDIT["frozen_main_design_conforming"] and FINAL_AUDIT["pilot_passed"] and FINAL_AUDIT["mnist_grid_exactly_complete"] and not DEBUG_MODE and not EMNIST_PROTOCOL_DEVIATION and (not PHASES["run_emnist_generalization_grid"] or FINAL_AUDIT["emnist_grid_exactly_complete"])) else "INCOMPLETE / PROTOCOL-DEVIATING / FINDINGS WITHHELD"
atomic_json(FINAL_AUDIT,EXPORT_ROOT/"final_audit.json")
display(pd.DataFrame([FINAL_AUDIT]).T.rename(columns={0:"value"}))
print("Exports:",EXPORT_ROOT)
# Refresh manifest after final_audit.json is created; the manifest intentionally excludes only itself.
artifacts=[]
for p in EXPORT_ROOT.rglob("*"):
    if p.is_file() and p.name!="artifact_manifest.csv":
        try: artifacts.append({"relative_path":str(p.relative_to(EXPORT_ROOT)),"bytes":p.stat().st_size,"sha256":sha256_file(p)})
        except Exception: pass
ARTIFACT_MANIFEST=pd.DataFrame(artifacts); ARTIFACT_MANIFEST.to_csv(EXPORT_ROOT/"artifact_manifest.csv",index=False)


,value
latent_dim_exactly_20,True
source_spec_sha256,7e60cb2f5db157122552f477c9f06d082ba5d97b380189...
training_implementation_fingerprint,41c0caf494b68e15488d5d5309d3d3bd5b955ca97574b0...
training_protocol_fingerprint,986f50280ca42fbee0b761f4603fe1f4ab41dd4df1fc31...
study_design_fingerprint,e20cd15d5b682615121ff80c4bc1202222dc5539adb2e7...
frozen_main_design_conforming,True
latent_sensitivity_excluded_by_current_z20_constraint,True
pilot_passed,True
mnist_grid_exactly_complete,True
emnist_grid_exactly_complete,True


Exports: /home/imohammedali/projects/pgm/latent_density_results/research_grade_z20_v2/sessions/20260908T052317_784076Z
